In [ ]:
#!/usr/bin/env python3
"""
Check whether input genes are reported in HIV/SIV and/or ART context.

Input:
    - A text file with one gene symbol per line, e.g.:
        IFNG
        IL6
        TNF
      (no header; blank lines are ignored)

For each gene:
    - Build a PubMed query:
        (gene in Title/Abstract or MeSH)
        AND (HIV/SIV OR antiretroviral therapy context)
    - Retrieve PMIDs
    - If ≥1 hit:
        - Fetch detailed metadata via EFetch (XML)
        - Pick the *earliest* paper by publication year
        - Build a BibTeX entry
    - If no hits:
        - Write a row with NA fields

Outputs:
    - Genes_HIV_SIV_ART_lit.csv
    - Genes_HIV_SIV_ART_lit.xlsx

Columns:
    Gene, PMID, Title, Journal, Year, DOI, FirstAuthor,
    Volume, Issue, Pages, PubMed_URL, BibTeX_Key, BibTeX,
    Query, Note
"""

import os
import time
import csv
import sys
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"   # REQUIRED by NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()  # optional but recommended

GENE_FILE = "/content/ltART_Baseline.txt"  # <-- your input file (one gene per line)

OUT_CSV  = "Genes_lt_B_lit.csv"
OUT_XLSX = "Genes_lt_B_lit.xlsx"

RETMAX = 100                        # max PMIDs to retrieve per gene
SLEEP_SECS = 0.35 if API_KEY else 0.6
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# HIV/SIV context – Title/Abstract and MeSH
HIV_TA_TERMS = [
    "HIV", "\"human immunodeficiency virus\"", "\"HIV-1\"", "\"HIV-2\"",
    "AIDS", "\"acquired immunodeficiency syndrome\"",
    "SIV", "\"simian immunodeficiency virus\"", "SHIV"
]

HIV_MESH_TERMS = [
    "HIV Infections[MeSH Terms]",
    "Acquired Immunodeficiency Syndrome[MeSH Terms]",
    "Simian Acquired Immunodeficiency Syndrome[MeSH Terms]"
]

# ART / treatment context – Title/Abstract terms
ART_TA_TERMS = [
    "\"antiretroviral therapy\"",
    "\"antiretroviral treatment\"",
    "\"ART regimen\"",
    "\"ART initiation\"",
    "HAART",
    "cART",
    "\"combined antiretroviral therapy\"",
    "\"highly active antiretroviral therapy\""
]

# Optional "short-term" / "long-term" flavour words (Title/Abstract)
SHORT_TERM_TERMS = [
    "\"short-term\"",
    "\"short term\"",
    "early",
    "acute"
]

LONG_TERM_TERMS = [
    "\"long-term\"",
    "\"long term\"",
    "chronic",
    "prolonged"
]


# -------------------- HELPERS --------------------
def read_genes(path):
    """Read gene symbols from a txt file (one per line)."""
    if not os.path.exists(path):
        print(f"❌ Gene file not found: {path}", file=sys.stderr)
        sys.exit(1)
    genes = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            term = line.strip()
            if term:
                genes.append(term)
    return genes


def build_query(gene: str) -> str:
    """
    Build PubMed query:
      (gene in Title/Abstract or MeSH)
      AND (HIV/SIV OR ART context)

    This captures:
      - gene + HIV/SIV (any stage)
      - gene + ART terms (HAART, cART, antiretroviral therapy, etc.)
      - includes words like 'short-term', 'long-term', 'acute', 'chronic' (soft flavour)
    """
    g = gene.replace('"', '')  # sanitize double quotes

    # gene in title/abstract or MeSH
    gene_block = (
        f"(\"{g}\"[Title/Abstract] "
        f"OR {g}[Title/Abstract] "
        f"OR \"{g}\"[MeSH Terms])"
    )

    hiv_ta   = "(" + " OR ".join(HIV_TA_TERMS) + ")"
    hiv_mesh = "(" + " OR ".join(HIV_MESH_TERMS) + ")"
    hiv_any  = f"({hiv_ta} OR {hiv_mesh})"

    art_any  = "(" + " OR ".join(ART_TA_TERMS) + ")"

    short_any = "(" + " OR ".join(SHORT_TERM_TERMS) + ")"
    long_any  = "(" + " OR ".join(LONG_TERM_TERMS) + ")"

    # ART-specific context block (not strictly required, but bias towards therapy papers)
    art_context = f"(({art_any}) OR (({art_any}) AND ({short_any} OR {long_any})))"

    # Final context: HIV/SIV OR ART in HIV/SIV setting
    # Loose: gene + (HIV/SIV OR ART). In practice ART is mostly HIV-related.
    context_block = f"({hiv_any} OR {art_context})"

    q = f"{gene_block} AND {context_block}"

    # Optional: restrict to more recent years (uncomment if you want)
    # q = f'{q} AND ("1990"[Date - Publication] : "3000"[Date - Publication])'
    return q


def eutils_request(path: str, params: dict, retries: int = 3):
    """Robust E-utilities call with retries and basic rate limiting."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS_BASE}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ E-utilities request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_SECS * attempt)


def esearch_pmids(query: str, retmax: int = 50):
    """Search PubMed and return up to retmax PMIDs."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": retmax,
        "retmode": "json",
        "sort": "relevance",
    }
    resp = eutils_request("esearch.fcgi", params).json()
    return resp.get("esearchresult", {}).get("idlist", [])


def pubmed_url(pmid: str) -> str:
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


def efetch_details(pmids):
    """
    Use PubMed EFetch (XML) to get metadata and BibTeX-ready info for a list of PMIDs.
    Returns dict[pmid] = {
        "Title", "Journal", "Year", "FirstAuthor", "DOI",
        "Volume", "Issue", "Pages",
        "BibTeX_Key", "BibTeX"
    }
    """
    if not pmids:
        return {}

    out = {}
    CHUNK_SIZE = 200
    for i in range(0, len(pmids), CHUNK_SIZE):
        chunk = pmids[i:i + CHUNK_SIZE]
        params = {
            "db": "pubmed",
            "id": ",".join(chunk),
            "retmode": "xml",
        }
        r = eutils_request("efetch.fcgi", params)
        root = ET.fromstring(r.text)

        for article in root.findall(".//PubmedArticle"):
            try:
                pmid_elem = article.find(".//PMID")
                pmid = pmid_elem.text.strip() if pmid_elem is not None else None
                if not pmid:
                    continue

                art = article.find(".//Article")
                if art is None:
                    continue

                # Title
                title_elem = art.find("ArticleTitle")
                title = "".join(title_elem.itertext()).strip() if title_elem is not None else ""

                # Journal + year, volume, issue, pages
                journal_elem = art.find("Journal")
                journal_title_elem = journal_elem.find("Title") if journal_elem is not None else None
                journal = journal_title_elem.text.strip() if journal_title_elem is not None else ""

                pub_date = journal_elem.find("JournalIssue/PubDate") if journal_elem is not None else None
                year = ""
                if pub_date is not None:
                    y = pub_date.find("Year")
                    if y is not None and y.text and y.text.strip().isdigit():
                        year = y.text.strip()
                    else:
                        md = pub_date.find("MedlineDate")
                        if md is not None and md.text:
                            for tok in md.text.split():
                                if tok.isdigit() and len(tok) == 4:
                                    year = tok
                                    break

                medline_pg = art.find("Pagination/MedlinePgn")
                pages = medline_pg.text.strip() if medline_pg is not None else ""
                vol = journal_elem.find("JournalIssue/Volume") if journal_elem is not None else None
                volume = vol.text.strip() if vol is not None else ""
                iss = journal_elem.find("JournalIssue/Issue") if journal_elem is not None else None
                issue = iss.text.strip() if iss is not None else ""

                # Authors
                author_list = art.findall("AuthorList/Author")
                authors = []
                first_author_lastname = ""
                for idx, au in enumerate(author_list):
                    last = au.find("LastName")
                    fore = au.find("ForeName")
                    initials = au.find("Initials")

                    if last is None or not last.text:
                        continue
                    last_name = last.text.strip()
                    if idx == 0:
                        first_author_lastname = last_name

                    if fore is not None and fore.text:
                        first_name = fore.text.strip()
                    elif initials is not None and initials.text:
                        first_name = initials.text.strip()
                    else:
                        first_name = ""
                    if first_name:
                        authors.append(f"{last_name}, {first_name}")
                    else:
                        authors.append(last_name)

                authors_str = " and ".join(authors)

                # DOI
                doi = ""
                for aid in article.findall(".//ArticleIdList/ArticleId"):
                    if aid.get("IdType") == "doi" and aid.text:
                        doi = aid.text.strip()
                        break

                # BibTeX key
                key_author = first_author_lastname if first_author_lastname else "UnknownAuthor"
                key_year = year if year else "n.d."
                journal_abbrev = journal.replace(" ", "")[:20]
                bib_key = f"{key_author}{key_year}_{journal_abbrev}"
                bib_key = "".join(ch for ch in bib_key if ch.isalnum() or ch in "_.-")

                def esc(s):
                    return s.replace("{", "").replace("}", "")

                bib_lines = [
                    f"@article{{{bib_key},",
                    f"  author  = {{{esc(authors_str)}}},",
                    f"  title   = {{{esc(title)}}},",
                    f"  journal = {{{esc(journal)}}},",
                    f"  year    = {{{year}}},",
                    f"  volume  = {{{volume}}},",
                    f"  number  = {{{issue}}},",
                    f"  pages   = {{{pages}}},",
                    f"  doi     = {{{doi}}},",
                    f"  pmid    = {{{pmid}}}",
                    "}"
                ]
                bibtex_str = "\n".join(bib_lines)

                out[pmid] = {
                    "Title": title,
                    "Journal": journal,
                    "Year": year,
                    "FirstAuthor": first_author_lastname,
                    "DOI": doi,
                    "Volume": volume,
                    "Issue": issue,
                    "Pages": pages,
                    "BibTeX_Key": bib_key,
                    "BibTeX": bibtex_str,
                }

            except Exception as e:
                print(f"⚠ Error parsing article in EFetch: {e}")
                continue

        time.sleep(SLEEP_SECS)

    return out


def pick_earliest_pmid(pmids, details_dict):
    """
    Given a list of pmids and a details dict (pmid -> Year, etc.),
    return the PMID with the earliest publication year.
    If year is missing or not numeric, treat as very late (9999).
    """
    if not pmids:
        return None

    def year_as_int(pmid):
        y = details_dict.get(pmid, {}).get("Year", "")
        if isinstance(y, int):
            return y
        try:
            return int(y)
        except (TypeError, ValueError):
            return 9999

    return min(pmids, key=year_as_int)


# -------------------- MAIN LOGIC --------------------
def main():
    if not NCBI_EMAIL or NCBI_EMAIL == "your_email@domain.com":
        print("❌ Please set your real email address in NCBI_EMAIL.", file=sys.stderr)
        sys.exit(1)

    genes = read_genes(GENE_FILE)
    print(f"📄 Loaded {len(genes)} genes from {GENE_FILE}\n")

    out_rows = []

    for i, gene in enumerate(genes, 1):
        print(f"[{i}/{len(genes)}] {gene} → PubMed search (HIV/SIV/ART context)")
        query = build_query(gene)
        try:
            pmids = esearch_pmids(query, retmax=RETMAX)
            time.sleep(SLEEP_SECS)

            if not pmids:
                print("   ⚠ No PubMed hits.")
                out_rows.append({
                    "Gene": gene,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "FirstAuthor": "NA",
                    "Volume": "NA",
                    "Issue": "NA",
                    "Pages": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "No PubMed result in HIV/SIV/ART context",
                })
                continue

            # Fetch metadata for all PMIDs, then pick earliest
            details = efetch_details(pmids)
            earliest_pmid = pick_earliest_pmid(pmids, details)

            if earliest_pmid is None or earliest_pmid not in details:
                print("   ⚠ Could not resolve earliest paper; writing NA row.")
                out_rows.append({
                    "Gene": gene,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "FirstAuthor": "NA",
                    "Volume": "NA",
                    "Issue": "NA",
                    "Pages": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "PMIDs found but metadata missing",
                })
                continue

            meta = details[earliest_pmid]
            print(f"   ✅ Earliest paper: PMID {earliest_pmid}, Year {meta.get('Year','')}")

            out_rows.append({
                "Gene": gene,
                "PMID": earliest_pmid,
                "Title": meta.get("Title", ""),
                "Journal": meta.get("Journal", ""),
                "Year": meta.get("Year", ""),
                "DOI": meta.get("DOI", ""),
                "FirstAuthor": meta.get("FirstAuthor", ""),
                "Volume": meta.get("Volume", ""),
                "Issue": meta.get("Issue", ""),
                "Pages": meta.get("Pages", ""),
                "PubMed_URL": pubmed_url(earliest_pmid),
                "BibTeX_Key": meta.get("BibTeX_Key", ""),
                "BibTeX": meta.get("BibTeX", ""),
                "Query": query,
                "Note": "",
            })

        except Exception as e:
            print(f"   ❌ Error for gene '{gene}': {e}")
            out_rows.append({
                "Gene": gene,
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "DOI": "NA",
                "FirstAuthor": "NA",
                "Volume": "NA",
                "Issue": "NA",
                "Pages": "NA",
                "PubMed_URL": "NA",
                "BibTeX_Key": "NA",
                "BibTeX": "NA",
                "Query": query,
                "Note": f"Error: {e}",
            })

        time.sleep(SLEEP_SECS)

    cols = [
        "Gene", "PMID", "Title", "Journal", "Year",
        "DOI", "FirstAuthor", "Volume", "Issue", "Pages",
        "PubMed_URL", "BibTeX_Key", "BibTeX",
        "Query", "Note"
    ]

    # CSV
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            # sanitize quotes inside strings
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    # Excel
    df = pd.DataFrame(out_rows, columns=cols)
    df.to_excel(OUT_XLSX, index=False)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


📄 Loaded 151 genes from /content/ltART_Baseline.txt

[1/151] Symbol → PubMed search (HIV/SIV/ART context)
   ✅ Earliest paper: PMID 1300378, Year 1992
[2/151] AATK → PubMed search (HIV/SIV/ART context)
   ✅ Earliest paper: PMID 25961023, Year 2015
[3/151] ABCC6 → PubMed search (HIV/SIV/ART context)
   ✅ Earliest paper: PMID 19075072, Year 2009
[4/151] AGAP1 → PubMed search (HIV/SIV/ART context)
   ✅ Earliest paper: PMID 40766577, Year 2025
[5/151] AGRN → PubMed search (HIV/SIV/ART context)
   ⚠ No PubMed hits.
[6/151] ALOXE3 → PubMed search (HIV/SIV/ART context)
   ⚠ No PubMed hits.
[7/151] ANGPTL6 → PubMed search (HIV/SIV/ART context)
   ⚠ No PubMed hits.
[8/151] C16H17orf97 → PubMed search (HIV/SIV/ART context)
   ⚠ No PubMed hits.
[9/151] C2H3orf86 → PubMed search (HIV/SIV/ART context)
   ⚠ No PubMed hits.
[10/151] C9H10orf105 → PubMed search (HIV/SIV/ART context)
   ⚠ No PubMed hits.
[11/151] CDCA5 → PubMed search (HIV/SIV/ART context)
   ✅ Earliest paper: PMID 28959563, Year 2016


In [ ]:
pip install requests pandas openpyxl

In [ ]:
#!/usr/bin/env python3
"""
Check whether PREART canonical pathways are reported in HIV/SIV literature.

Input:
    - A text file with one pathway term per line, e.g.:
        Interferon signaling
        Neuroinflammation signaling pathway
        VEGF signaling
      (no header; blank lines are ignored)

For each pathway:
    - Build a PubMed query:
        (pathway term in Title/Abstract) AND (HIV/SIV context terms)
    - Retrieve PMIDs
    - If ≥1 hit:
        - Fetch detailed metadata via EFetch (XML)
        - Pick the *earliest* paper by publication year
        - Build a BibTeX entry
    - If no hits:
        - Write a row with NA fields

Outputs:
    - SIV_PREART_pathways_lit.csv
    - SIV_PREART_pathways_lit.xlsx

Columns:
    Pathway, PMID, Title, Journal, Year, DOI, PubMed_URL,
    BibTeX_Key, BibTeX, Query, Note
"""

import os
import time
import csv
import sys
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"   # REQUIRED by NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()  # optional but recommended

PATHWAY_FILE = "PreART_pathways.txt"  # <-- your input file (one pathway per line)

OUT_CSV  = "SIV_PREART_pathways_lit.csv"
OUT_XLSX = "SIV_PREART_pathways_lit.xlsx"

RETMAX = 100                        # max PMIDs to retrieve per pathway
SLEEP_SECS = 0.35 if API_KEY else 0.6
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# HIV/SIV context – Title/Abstract and MeSH
HIV_TA_TERMS = [
    "HIV", "\"human immunodeficiency virus\"", "\"HIV-1\"", "\"HIV-2\"",
    "AIDS", "\"acquired immunodeficiency syndrome\"",
    "SIV", "\"simian immunodeficiency virus\"", "SHIV"
]

HIV_MESH_TERMS = [
    "HIV Infections[MeSH Terms]",
    "Acquired Immunodeficiency Syndrome[MeSH Terms]",
    "Simian Acquired Immunodeficiency Syndrome[MeSH Terms]"
]


# -------------------- HELPERS --------------------
def read_pathways(path):
    """Read pathway names from a txt file (one per line)."""
    if not os.path.exists(path):
        print(f"❌ Pathway file not found: {path}", file=sys.stderr)
        sys.exit(1)
    pathways = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            term = line.strip()
            if term:
                pathways.append(term)
    return pathways


def build_query(pathway: str) -> str:
    """
    Build PubMed query:
      (pathway term in Title/Abstract)
      AND (HIV/SIV context in Title/Abstract or MeSH)
    """
    t = pathway.replace('"', '')  # sanitize double quotes
    term_block = f"(\"{t}\"[Title/Abstract] OR {t}[Title/Abstract])"

    hiv_ta  = "(" + " OR ".join(HIV_TA_TERMS) + ")"
    hiv_mesh = "(" + " OR ".join(HIV_MESH_TERMS) + ")"
    hiv_any = f"({hiv_ta} OR {hiv_mesh})"

    q = f"{term_block} AND {hiv_any}"

    # Optional: restrict to more recent years (uncomment if you want)
    # q = f'{q} AND ("1990"[Date - Publication] : "3000"[Date - Publication])'
    return q


def eutils_request(path: str, params: dict, retries: int = 3):
    """Robust E-utilities call with retries and basic rate limiting."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS_BASE}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ E-utilities request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_SECS * attempt)


def esearch_pmids(query: str, retmax: int = 50):
    """Search PubMed and return up to retmax PMIDs."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": retmax,
        "retmode": "json",
        "sort": "relevance",
    }
    resp = eutils_request("esearch.fcgi", params).json()
    return resp.get("esearchresult", {}).get("idlist", [])


def pubmed_url(pmid: str) -> str:
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


def efetch_details(pmids):
    """
    Use PubMed EFetch (XML) to get metadata and BibTeX-ready info for a list of PMIDs.
    Returns dict[pmid] = {
        "Title", "Journal", "Year", "FirstAuthor", "DOI",
        "Volume", "Issue", "Pages",
        "BibTeX_Key", "BibTeX"
    }
    """
    if not pmids:
        return {}

    out = {}
    CHUNK_SIZE = 200
    for i in range(0, len(pmids), CHUNK_SIZE):
        chunk = pmids[i:i + CHUNK_SIZE]
        params = {
            "db": "pubmed",
            "id": ",".join(chunk),
            "retmode": "xml",
        }
        r = eutils_request("efetch.fcgi", params)
        root = ET.fromstring(r.text)

        for article in root.findall(".//PubmedArticle"):
            try:
                pmid_elem = article.find(".//PMID")
                pmid = pmid_elem.text.strip() if pmid_elem is not None else None
                if not pmid:
                    continue

                art = article.find(".//Article")
                if art is None:
                    continue

                # Title
                title_elem = art.find("ArticleTitle")
                title = "".join(title_elem.itertext()).strip() if title_elem is not None else ""

                # Journal + year, volume, issue, pages
                journal_elem = art.find("Journal")
                journal_title_elem = journal_elem.find("Title") if journal_elem is not None else None
                journal = journal_title_elem.text.strip() if journal_title_elem is not None else ""

                pub_date = journal_elem.find("JournalIssue/PubDate") if journal_elem is not None else None
                year = ""
                if pub_date is not None:
                    y = pub_date.find("Year")
                    if y is not None and y.text and y.text.strip().isdigit():
                        year = y.text.strip()
                    else:
                        md = pub_date.find("MedlineDate")
                        if md is not None and md.text:
                            for tok in md.text.split():
                                if tok.isdigit() and len(tok) == 4:
                                    year = tok
                                    break

                medline_pg = art.find("Pagination/MedlinePgn")
                pages = medline_pg.text.strip() if medline_pg is not None else ""
                vol = art.find("JournalIssue/Volume")
                volume = vol.text.strip() if vol is not None else ""
                iss = art.find("JournalIssue/Issue")
                issue = iss.text.strip() if iss is not None else ""

                # Authors
                author_list = art.findall("AuthorList/Author")
                authors = []
                first_author_lastname = ""
                for idx, au in enumerate(author_list):
                    last = au.find("LastName")
                    fore = au.find("ForeName")
                    initials = au.find("Initials")

                    if last is None or not last.text:
                        continue
                    last_name = last.text.strip()
                    if idx == 0:
                        first_author_lastname = last_name

                    if fore is not None and fore.text:
                        first_name = fore.text.strip()
                    elif initials is not None and initials.text:
                        first_name = initials.text.strip()
                    else:
                        first_name = ""
                    if first_name:
                        authors.append(f"{last_name}, {first_name}")
                    else:
                        authors.append(last_name)

                authors_str = " and ".join(authors)

                # DOI
                doi = ""
                for aid in article.findall(".//ArticleIdList/ArticleId"):
                    if aid.get("IdType") == "doi" and aid.text:
                        doi = aid.text.strip()
                        break

                # BibTeX key
                key_author = first_author_lastname if first_author_lastname else "UnknownAuthor"
                key_year = year if year else "n.d."
                journal_abbrev = journal.replace(" ", "")[:20]
                bib_key = f"{key_author}{key_year}_{journal_abbrev}"
                bib_key = "".join(ch for ch in bib_key if ch.isalnum() or ch in "_.-")

                def esc(s):
                    return s.replace("{", "").replace("}", "")

                bib_lines = [
                    f"@article{{{bib_key},",
                    f"  author  = {{{esc(authors_str)}}},",
                    f"  title   = {{{esc(title)}}},",
                    f"  journal = {{{esc(journal)}}},",
                    f"  year    = {{{year}}},",
                    f"  volume  = {{{volume}}},",
                    f"  number  = {{{issue}}},",
                    f"  pages   = {{{pages}}},",
                    f"  doi     = {{{doi}}},",
                    f"  pmid    = {{{pmid}}}",
                    "}"
                ]
                bibtex_str = "\n".join(bib_lines)

                out[pmid] = {
                    "Title": title,
                    "Journal": journal,
                    "Year": year,
                    "FirstAuthor": first_author_lastname,
                    "DOI": doi,
                    "Volume": volume,
                    "Issue": issue,
                    "Pages": pages,
                    "BibTeX_Key": bib_key,
                    "BibTeX": bibtex_str,
                }

            except Exception as e:
                print(f"⚠ Error parsing article in EFetch: {e}")
                continue

        time.sleep(SLEEP_SECS)

    return out


def pick_earliest_pmid(pmids, details_dict):
    """
    Given a list of pmids and a details dict (pmid -> Year, etc.),
    return the PMID with the earliest publication year.
    If year is missing or not numeric, treat as very late (9999).
    """
    if not pmids:
        return None

    def year_as_int(pmid):
        y = details_dict.get(pmid, {}).get("Year", "")
        if isinstance(y, int):
            return y
        try:
            return int(y)
        except (TypeError, ValueError):
            return 9999

    return min(pmids, key=year_as_int)


# -------------------- MAIN LOGIC --------------------
def main():
    if not NCBI_EMAIL or NCBI_EMAIL == "your_email@domain.com":
        print("❌ Please set your real email address in NCBI_EMAIL.", file=sys.stderr)
        sys.exit(1)

    pathways = read_pathways(PATHWAY_FILE)
    print(f"📄 Loaded {len(pathways)} pathways from {PATHWAY_FILE}\n")

    out_rows = []

    for i, pathway in enumerate(pathways, 1):
        print(f"[{i}/{len(pathways)}] {pathway} → PubMed search (HIV/SIV context)")
        query = build_query(pathway)
        try:
            pmids = esearch_pmids(query, retmax=RETMAX)
            time.sleep(SLEEP_SECS)

            if not pmids:
                print("   ⚠ No PubMed hits.")
                out_rows.append({
                    "Pathway": pathway,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "No PubMed result in HIV/SIV context",
                })
                continue

            # Fetch metadata for all PMIDs, then pick earliest
            details = efetch_details(pmids)
            earliest_pmid = pick_earliest_pmid(pmids, details)

            if earliest_pmid is None or earliest_pmid not in details:
                print("   ⚠ Could not resolve earliest paper; writing NA row.")
                out_rows.append({
                    "Pathway": pathway,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "PMIDs found but metadata missing",
                })
                continue

            meta = details[earliest_pmid]
            print(f"   ✅ Earliest paper: PMID {earliest_pmid}, Year {meta.get('Year','')}")

            out_rows.append({
                "Pathway": pathway,
                "PMID": earliest_pmid,
                "Title": meta.get("Title", ""),
                "Journal": meta.get("Journal", ""),
                "Year": meta.get("Year", ""),
                "DOI": meta.get("DOI", ""),
                "PubMed_URL": pubmed_url(earliest_pmid),
                "BibTeX_Key": meta.get("BibTeX_Key", ""),
                "BibTeX": meta.get("BibTeX", ""),
                "Query": query,
                "Note": "",
            })

        except Exception as e:
            print(f"   ❌ Error for pathway '{pathway}': {e}")
            out_rows.append({
                "Pathway": pathway,
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "DOI": "NA",
                "PubMed_URL": "NA",
                "BibTeX_Key": "NA",
                "BibTeX": "NA",
                "Query": query,
                "Note": f"Error: {e}",
            })

        time.sleep(SLEEP_SECS)

    cols = [
        "Pathway", "PMID", "Title", "Journal", "Year",
        "DOI", "PubMed_URL", "BibTeX_Key", "BibTeX",
        "Query", "Note"
    ]

    # CSV
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            # sanitize quotes inside strings
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    # Excel
    df = pd.DataFrame(out_rows, columns=cols)
    df.to_excel(OUT_XLSX, index=False)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


📄 Loaded 883 pathways from PreART_pathways.txt

[1/883] preart_baseline_cp → PubMed search (HIV/SIV context)
   ⚠ No PubMed hits.
[2/883] Interferon alpha/beta signaling → PubMed search (HIV/SIV context)
   ⚠ No PubMed hits.
[3/883] Interferon gamma signaling → PubMed search (HIV/SIV context)
   ✅ Earliest paper: PMID 29582356, Year 2018
[4/883] ISG15 antiviral mechanism → PubMed search (HIV/SIV context)
   ⚠ No PubMed hits.
[5/883] Multiple Sclerosis Signaling Pathway → PubMed search (HIV/SIV context)
   ✅ Earliest paper: PMID 14730712, Year 2004
[6/883] Role of Hypercytokinemia/hyperchemokinemia in the Pathogenesis of Influenza → PubMed search (HIV/SIV context)
   ⚠ No PubMed hits.
[7/883] NOD1/2 Signaling Pathway → PubMed search (HIV/SIV context)
   ⚠ No PubMed hits.
[8/883] Pathogen Induced Cytokine Storm Signaling Pathway → PubMed search (HIV/SIV context)
   ⚠ No PubMed hits.
[9/883] Parkinson's Signaling Pathway → PubMed search (HIV/SIV context)
   ✅ Earliest paper: PMID 12393944

In [ ]:
#!/usr/bin/env python3
"""
Check whether canonical pathways are reported as affected after
short-term antiretroviral therapy (ART) in HIV/SIV literature.

Input:
    - A text file with one pathway term per line, e.g.:
        Interferon signaling
        Neuroinflammation signaling pathway
        VEGF signaling
      (no header; blank lines are ignored)

For each pathway:
    - Build a PubMed query:
        (pathway term in Title/Abstract)
        AND (HIV/SIV context terms)
        AND (ART / HAART / cART / ART initiation / early ART terms)
    - Retrieve PMIDs
    - If ≥1 hit:
        - Fetch detailed metadata via EFetch (XML)
        - Pick the *earliest* paper by publication year
        - Build a BibTeX entry
    - If no hits:
        - Write a row with NA fields

Outputs:
    - SIV_stART_pathways_lit.csv
    - SIV_stART_pathways_lit.xlsx

Columns:
    Pathway, PMID, Title, Journal, Year, DOI, PubMed_URL,
    BibTeX_Key, BibTeX, Query, Note
"""

import os
import time
import csv
import sys
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"   # REQUIRED by NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()  # optional but recommended

# <-- your input file for stART pathways (one pathway per line)
PATHWAY_FILE = "stART_pathways.txt"

OUT_CSV  = "SIV_stART_pathways_lit.csv"
OUT_XLSX = "SIV_stART_pathways_lit.xlsx"

RETMAX = 100                        # max PMIDs to retrieve per pathway
SLEEP_SECS = 0.35 if API_KEY else 0.6
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# HIV/SIV context – Title/Abstract and MeSH
HIV_TA_TERMS = [
    "HIV", "\"human immunodeficiency virus\"", "\"HIV-1\"", "\"HIV-2\"",
    "AIDS", "\"acquired immunodeficiency syndrome\"",
    "SIV", "\"simian immunodeficiency virus\"", "SHIV"
]

HIV_MESH_TERMS = [
    "HIV Infections[MeSH Terms]",
    "Acquired Immunodeficiency Syndrome[MeSH Terms]",
    "Simian Acquired Immunodeficiency Syndrome[MeSH Terms]"
]

# ART / treatment context – Title/Abstract and MeSH
ART_TA_TERMS = [
    "\"antiretroviral therapy\"",
    "\"antiretroviral treatment\"",
    "\"HAART\"",
    "\"highly active antiretroviral therapy\"",
    "\"cART\"",
    "\"combined antiretroviral therapy\"",
    "\"ART initiation\"",
    "\"early ART\"",
    "\"short-term ART\"",
    "\"short term ART\"",
    "\"treatment-naive\"",
    "\"treatment naive\"",
    "\"ART-treated\"",
    "\"ART treated\""
]

ART_MESH_TERMS = [
    "Anti-Retroviral Agents[MeSH Terms]",
    "Anti-HIV Agents[MeSH Terms]",
    "Antiretroviral Therapy, Highly Active[MeSH Terms]"
]


# -------------------- HELPERS --------------------
def read_pathways(path):
    """Read pathway names from a txt file (one per line)."""
    if not os.path.exists(path):
        print(f"❌ Pathway file not found: {path}", file=sys.stderr)
        sys.exit(1)
    pathways = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            term = line.strip()
            if term:
                pathways.append(term)
    return pathways


def build_query(pathway: str) -> str:
    """
    Build PubMed query:

      (pathway term in Title/Abstract)
      AND (HIV/SIV context in Title/Abstract or MeSH)
      AND (ART / HAART / cART context in Title/Abstract or MeSH)
    """
    t = pathway.replace('"', '')  # sanitize double quotes
    term_block = f"(\"{t}\"[Title/Abstract] OR {t}[Title/Abstract])"

    hiv_ta   = "(" + " OR ".join(HIV_TA_TERMS) + ")"
    hiv_mesh = "(" + " OR ".join(HIV_MESH_TERMS) + ")"
    hiv_any  = f"({hiv_ta} OR {hiv_mesh})"

    art_ta   = "(" + " OR ".join(ART_TA_TERMS) + ")"
    art_mesh = "(" + " OR ".join(ART_MESH_TERMS) + ")"
    art_any  = f"({art_ta} OR {art_mesh})"

    # Require BOTH HIV/SIV AND ART context
    q = f"{term_block} AND {hiv_any} AND {art_any}"

    # Optional: restrict to more recent years (uncomment if you want)
    # q = f'{q} AND ("1996"[Date - Publication] : "3000"[Date - Publication])'
    return q


def eutils_request(path: str, params: dict, retries: int = 3):
    """Robust E-utilities call with retries and basic rate limiting."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS_BASE}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ E-utilities request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_SECS * attempt)


def esearch_pmids(query: str, retmax: int = 50):
    """Search PubMed and return up to retmax PMIDs."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": retmax,
        "retmode": "json",
        "sort": "relevance",
    }
    resp = eutils_request("esearch.fcgi", params).json()
    return resp.get("esearchresult", {}).get("idlist", [])


def pubmed_url(pmid: str) -> str:
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


def efetch_details(pmids):
    """
    Use PubMed EFetch (XML) to get metadata and BibTeX-ready info for a list of PMIDs.
    Returns dict[pmid] = {
        "Title", "Journal", "Year", "FirstAuthor", "DOI",
        "Volume", "Issue", "Pages",
        "BibTeX_Key", "BibTeX"
    }
    """
    if not pmids:
        return {}

    out = {}
    CHUNK_SIZE = 200
    for i in range(0, len(pmids), CHUNK_SIZE):
        chunk = pmids[i:i + CHUNK_SIZE]
        params = {
            "db": "pubmed",
            "id": ",".join(chunk),
            "retmode": "xml",
        }
        r = eutils_request("efetch.fcgi", params)
        root = ET.fromstring(r.text)

        for article in root.findall(".//PubmedArticle"):
            try:
                pmid_elem = article.find(".//PMID")
                pmid = pmid_elem.text.strip() if pmid_elem is not None else None
                if not pmid:
                    continue

                art = article.find(".//Article")
                if art is None:
                    continue

                # Title
                title_elem = art.find("ArticleTitle")
                title = "".join(title_elem.itertext()).strip() if title_elem is not None else ""

                # Journal + year, volume, issue, pages
                journal_elem = art.find("Journal")
                journal_title_elem = journal_elem.find("Title") if journal_elem is not None else None
                journal = journal_title_elem.text.strip() if journal_title_elem is not None else ""

                pub_date = journal_elem.find("JournalIssue/PubDate") if journal_elem is not None else None
                year = ""
                if pub_date is not None:
                    y = pub_date.find("Year")
                    if y is not None and y.text and y.text.strip().isdigit():
                        year = y.text.strip()
                    else:
                        md = pub_date.find("MedlineDate")
                        if md is not None and md.text:
                            for tok in md.text.split():
                                if tok.isdigit() and len(tok) == 4:
                                    year = tok
                                    break

                medline_pg = art.find("Pagination/MedlinePgn")
                pages = medline_pg.text.strip() if medline_pg is not None else ""
                vol = art.find("JournalIssue/Volume")
                volume = vol.text.strip() if vol is not None else ""
                iss = art.find("JournalIssue/Issue")
                issue = iss.text.strip() if iss is not None else ""

                # Authors
                author_list = art.findall("AuthorList/Author")
                authors = []
                first_author_lastname = ""
                for idx, au in enumerate(author_list):
                    last = au.find("LastName")
                    fore = au.find("ForeName")
                    initials = au.find("Initials")

                    if last is None or not last.text:
                        continue
                    last_name = last.text.strip()
                    if idx == 0:
                        first_author_lastname = last_name

                    if fore is not None and fore.text:
                        first_name = fore.text.strip()
                    elif initials is not None and initials.text:
                        first_name = initials.text.strip()
                    else:
                        first_name = ""
                    if first_name:
                        authors.append(f"{last_name}, {first_name}")
                    else:
                        authors.append(last_name)

                authors_str = " and ".join(authors)

                # DOI
                doi = ""
                for aid in article.findall(".//ArticleIdList/ArticleId"):
                    if aid.get("IdType") == "doi" and aid.text:
                        doi = aid.text.strip()
                        break

                # BibTeX key
                key_author = first_author_lastname if first_author_lastname else "UnknownAuthor"
                key_year = year if year else "n.d."
                journal_abbrev = journal.replace(" ", "")[:20]
                bib_key = f"{key_author}{key_year}_{journal_abbrev}"
                bib_key = "".join(ch for ch in bib_key if ch.isalnum() or ch in "_.-")

                def esc(s):
                    return s.replace("{", "").replace("}", "")

                bib_lines = [
                    f"@article{{{bib_key},",
                    f"  author  = {{{esc(authors_str)}}},",
                    f"  title   = {{{esc(title)}}},",
                    f"  journal = {{{esc(journal)}}},",
                    f"  year    = {{{year}}},",
                    f"  volume  = {{{volume}}},",
                    f"  number  = {{{issue}}},",
                    f"  pages   = {{{pages}}},",
                    f"  doi     = {{{doi}}},",
                    f"  pmid    = {{{pmid}}}",
                    "}"
                ]
                bibtex_str = "\n".join(bib_lines)

                out[pmid] = {
                    "Title": title,
                    "Journal": journal,
                    "Year": year,
                    "FirstAuthor": first_author_lastname,
                    "DOI": doi,
                    "Volume": volume,
                    "Issue": issue,
                    "Pages": pages,
                    "BibTeX_Key": bib_key,
                    "BibTeX": bibtex_str,
                }

            except Exception as e:
                print(f"⚠ Error parsing article in EFetch: {e}")
                continue

        time.sleep(SLEEP_SECS)

    return out


def pick_earliest_pmid(pmids, details_dict):
    """
    Given a list of pmids and a details dict (pmid -> Year, etc.),
    return the PMID with the earliest publication year.
    If year is missing or not numeric, treat as very late (9999).
    """
    if not pmids:
        return None

    def year_as_int(pmid):
        y = details_dict.get(pmid, {}).get("Year", "")
        if isinstance(y, int):
            return y
        try:
            return int(y)
        except (TypeError, ValueError):
            return 9999

    return min(pmids, key=year_as_int)


# -------------------- MAIN LOGIC --------------------
def main():
    if not NCBI_EMAIL or NCBI_EMAIL == "your_email@domain.com":
        print("❌ Please set your real email address in NCBI_EMAIL.", file=sys.stderr)
        sys.exit(1)

    pathways = read_pathways(PATHWAY_FILE)
    print(f"📄 Loaded {len(pathways)} pathways from {PATHWAY_FILE}\n")

    out_rows = []

    for i, pathway in enumerate(pathways, 1):
        print(f"[{i}/{len(pathways)}] {pathway} → PubMed search (HIV/SIV + ART context)")
        query = build_query(pathway)
        try:
            pmids = esearch_pmids(query, retmax=RETMAX)
            time.sleep(SLEEP_SECS)

            if not pmids:
                print("   ⚠ No PubMed hits.")
                out_rows.append({
                    "Pathway": pathway,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "No PubMed result in HIV/SIV + ART context",
                })
                continue

            # Fetch metadata for all PMIDs, then pick earliest
            details = efetch_details(pmids)
            earliest_pmid = pick_earliest_pmid(pmids, details)

            if earliest_pmid is None or earliest_pmid not in details:
                print("   ⚠ Could not resolve earliest paper; writing NA row.")
                out_rows.append({
                    "Pathway": pathway,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "PMIDs found but metadata missing",
                })
                continue

            meta = details[earliest_pmid]
            print(f"   ✅ Earliest paper: PMID {earliest_pmid}, Year {meta.get('Year','')}")

            out_rows.append({
                "Pathway": pathway,
                "PMID": earliest_pmid,
                "Title": meta.get("Title", ""),
                "Journal": meta.get("Journal", ""),
                "Year": meta.get("Year", ""),
                "DOI": meta.get("DOI", ""),
                "PubMed_URL": pubmed_url(earliest_pmid),
                "BibTeX_Key": meta.get("BibTeX_Key", ""),
                "BibTeX": meta.get("BibTeX", ""),
                "Query": query,
                "Note": "",
            })

        except Exception as e:
            print(f"   ❌ Error for pathway '{pathway}': {e}")
            out_rows.append({
                "Pathway": pathway,
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "DOI": "NA",
                "PubMed_URL": "NA",
                "BibTeX_Key": "NA",
                "BibTeX": "NA",
                "Query": query,
                "Note": f"Error: {e}",
            })

        time.sleep(SLEEP_SECS)

    cols = [
        "Pathway", "PMID", "Title", "Journal", "Year",
        "DOI", "PubMed_URL", "BibTeX_Key", "BibTeX",
        "Query", "Note"
    ]

    # CSV
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            # sanitize quotes inside strings
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    # Excel
    df = pd.DataFrame(out_rows, columns=cols)
    df.to_excel(OUT_XLSX, index=False)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


📄 Loaded 69 pathways from stART_pathways.txt

[1/69] CP → PubMed search (HIV/SIV + ART context)
   ✅ Earliest paper: PMID 10839662, Year 2000
[2/69] Oxytocin Signaling Pathway → PubMed search (HIV/SIV + ART context)
   ⚠ No PubMed hits.
[3/69] Orexin Signaling Pathway → PubMed search (HIV/SIV + ART context)
   ⚠ No PubMed hits.
[4/69] Cardiac Hypertrophy Signaling (Enhanced) → PubMed search (HIV/SIV + ART context)
   ✅ Earliest paper: PMID 18222550, Year 2009
[5/69] Hepatic Fibrosis Signaling Pathway → PubMed search (HIV/SIV + ART context)
   ⚠ No PubMed hits.
[6/69] Multiple Sclerosis Signaling Pathway → PubMed search (HIV/SIV + ART context)
   ⚠ No PubMed hits.
[7/69] Erythrocytes take up carbon dioxide and release oxygen → PubMed search (HIV/SIV + ART context)
   ⚠ No PubMed hits.
[8/69] Erythrocytes take up oxygen and release carbon dioxide → PubMed search (HIV/SIV + ART context)
   ⚠ No PubMed hits.
[9/69] Interferon gamma signaling → PubMed search (HIV/SIV + ART context)
   ✅ Ear

In [ ]:
#!/usr/bin/env python3
"""
Check whether canonical pathways are reported as affected under
LONG-TERM antiretroviral therapy (ART) in HIV/SIV literature.

Specifically, this helps you see if pathways are known to be altered in:
  - long-term ART–treated HIV/SIV-infected individuals
  - often compared to healthy / HIV-negative / uninfected controls.

Input:
    - A text file with one pathway term per line, e.g.:
        Interferon signaling
        Neuroinflammation signaling pathway
        VEGF signaling
      (no header; blank lines are ignored)

For each pathway:
    - Build a PubMed query:
        (pathway term in Title/Abstract)
        AND (HIV/SIV context terms)
        AND (long-term ART / chronic ART / virally suppressed on ART terms)
    - Retrieve PMIDs
    - If ≥1 hit:
        - Fetch detailed metadata via EFetch (XML)
        - Pick the *earliest* paper by publication year
        - Build a BibTeX entry
    - If no hits:
        - Write a row with NA fields

Outputs:
    - SIV_ltART_pathways_lit.csv
    - SIV_ltART_pathways_lit.xlsx

Columns:
    Pathway, PMID, Title, Journal, Year, DOI, PubMed_URL,
    BibTeX_Key, BibTeX, Query, Note
"""

import os
import time
import csv
import sys
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"   # REQUIRED by NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()  # optional but recommended

# <-- your input file for LT-ART pathways (one pathway per line)
PATHWAY_FILE = "ltART_pathways.txt"

OUT_CSV  = "SIV_ltART_pathways_lit.csv"
OUT_XLSX = "SIV_ltART_pathways_lit.xlsx"

RETMAX = 100                        # max PMIDs to retrieve per pathway
SLEEP_SECS = 0.35 if API_KEY else 0.6
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# HIV/SIV context – Title/Abstract and MeSH
HIV_TA_TERMS = [
    "HIV", "\"human immunodeficiency virus\"", "\"HIV-1\"", "\"HIV-2\"",
    "AIDS", "\"acquired immunodeficiency syndrome\"",
    "SIV", "\"simian immunodeficiency virus\"", "SHIV"
]

HIV_MESH_TERMS = [
    "HIV Infections[MeSH Terms]",
    "Acquired Immunodeficiency Syndrome[MeSH Terms]",
    "Simian Acquired Immunodeficiency Syndrome[MeSH Terms]"
]

# LONG-TERM ART / treatment context – Title/Abstract and MeSH
# (Includes generic ART terms + long-term / chronic / virally suppressed)
ART_TA_TERMS = [
    # long-term / chronic ART phrases
    "\"long-term antiretroviral therapy\"",
    "\"long term antiretroviral therapy\"",
    "\"long-term ART\"",
    "\"long term ART\"",
    "\"long-term HAART\"",
    "\"chronic HIV on ART\"",
    "\"chronically HIV-infected on ART\"",
    "\"chronically HIV infected on ART\"",
    "\"long-term cART\"",
    "\"long term cART\"",
    "\"prolonged ART\"",
    "\"sustained ART\"",
    "\"sustained virologic suppression\"",
    "\"sustained viral suppression\"",
    "\"virally suppressed on ART\"",
    "\"virally suppressed HIV\"",
    "\"ART-suppressed\"",
    "\"ART suppressed\"",
    "\"treated HIV infection\"",

    # keep generic ART terms as well
    "\"antiretroviral therapy\"",
    "\"antiretroviral treatment\"",
    "\"HAART\"",
    "\"highly active antiretroviral therapy\"",
    "\"cART\"",
    "\"combined antiretroviral therapy\"",
    "\"ART-treated\"",
    "\"ART treated\""
]

ART_MESH_TERMS = [
    "Anti-Retroviral Agents[MeSH Terms]",
    "Anti-HIV Agents[MeSH Terms]",
    "Antiretroviral Therapy, Highly Active[MeSH Terms]"
]

# Optional helper terms describing comparisons vs healthy controls.
# We will NOT hard-AND these (to avoid making queries too strict),
# but they are included in ART_TA_TERMS-like phrases above in practice.
HEALTHY_CONTROL_TA_TERMS = [
    "\"healthy controls\"",
    "\"HIV-negative\"",
    "\"HIV negative\"",
    "\"HIV-uninfected\"",
    "\"HIV uninfected\"",
    "\"seronegative controls\"",
    "\"uninfected controls\""
]


# -------------------- HELPERS --------------------
def read_pathways(path):
    """Read pathway names from a txt file (one per line)."""
    if not os.path.exists(path):
        print(f"❌ Pathway file not found: {path}", file=sys.stderr)
        sys.exit(1)
    pathways = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            term = line.strip()
            if term:
                pathways.append(term)
    return pathways


def build_query(pathway: str) -> str:
    """
    Build PubMed query:

      (pathway term in Title/Abstract)
      AND (HIV/SIV context in Title/Abstract or MeSH)
      AND (long-term / chronic ART context in Title/Abstract or MeSH)

    This tends to retrieve:
      - long-term ART-treated HIV/SIV cohorts,
      - often in comparison to healthy / HIV-negative controls.
    """
    t = pathway.replace('"', '')  # sanitize double quotes
    term_block = f"(\"{t}\"[Title/Abstract] OR {t}[Title/Abstract])"

    hiv_ta   = "(" + " OR ".join(HIV_TA_TERMS) + ")"
    hiv_mesh = "(" + " OR ".join(HIV_MESH_TERMS) + ")"
    hiv_any  = f"({hiv_ta} OR {hiv_mesh})"

    art_ta   = "(" + " OR ".join(ART_TA_TERMS) + ")"
    art_mesh = "(" + " OR ".join(ART_MESH_TERMS) + ")"
    art_any  = f"({art_ta} OR {art_mesh})"

    # Require BOTH HIV/SIV AND ART context
    q = f"{term_block} AND {hiv_any} AND {art_any}"

    # Optional: restrict to ART era (post-1996)
    # q = f'{q} AND ("1996"[Date - Publication] : "3000"[Date - Publication])'

    return q


def eutils_request(path: str, params: dict, retries: int = 3):
    """Robust E-utilities call with retries and basic rate limiting."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS_BASE}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ E-utilities request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_SECS * attempt)


def esearch_pmids(query: str, retmax: int = 50):
    """Search PubMed and return up to retmax PMIDs."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": retmax,
        "retmode": "json",
        "sort": "relevance",
    }
    resp = eutils_request("esearch.fcgi", params).json()
    return resp.get("esearchresult", {}).get("idlist", [])


def pubmed_url(pmid: str) -> str:
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


def efetch_details(pmids):
    """
    Use PubMed EFetch (XML) to get metadata and BibTeX-ready info for a list of PMIDs.
    Returns dict[pmid] = {
        "Title", "Journal", "Year", "FirstAuthor", "DOI",
        "Volume", "Issue", "Pages",
        "BibTeX_Key", "BibTeX"
    }
    """
    if not pmids:
        return {}

    out = {}
    CHUNK_SIZE = 200
    for i in range(0, len(pmids), CHUNK_SIZE):
        chunk = pmids[i:i + CHUNK_SIZE]
        params = {
            "db": "pubmed",
            "id": ",".join(chunk),
            "retmode": "xml",
        }
        r = eutils_request("efetch.fcgi", params)
        root = ET.fromstring(r.text)

        for article in root.findall(".//PubmedArticle"):
            try:
                pmid_elem = article.find(".//PMID")
                pmid = pmid_elem.text.strip() if pmid_elem is not None else None
                if not pmid:
                    continue

                art = article.find(".//Article")
                if art is None:
                    continue

                # Title
                title_elem = art.find("ArticleTitle")
                title = "".join(title_elem.itertext()).strip() if title_elem is not None else ""

                # Journal + year, volume, issue, pages
                journal_elem = art.find("Journal")
                journal_title_elem = journal_elem.find("Title") if journal_elem is not None else None
                journal = journal_title_elem.text.strip() if journal_title_elem is not None else ""

                pub_date = journal_elem.find("JournalIssue/PubDate") if journal_elem is not None else None
                year = ""
                if pub_date is not None:
                    y = pub_date.find("Year")
                    if y is not None and y.text and y.text.strip().isdigit():
                        year = y.text.strip()
                    else:
                        md = pub_date.find("MedlineDate")
                        if md is not None and md.text:
                            for tok in md.text.split():
                                if tok.isdigit() and len(tok) == 4:
                                    year = tok
                                    break

                medline_pg = art.find("Pagination/MedlinePgn")
                pages = medline_pg.text.strip() if medline_pg is not None else ""
                vol = art.find("JournalIssue/Volume")
                volume = vol.text.strip() if vol is not None else ""
                iss = art.find("JournalIssue/Issue")
                issue = iss.text.strip() if iss is not None else ""

                # Authors
                author_list = art.findall("AuthorList/Author")
                authors = []
                first_author_lastname = ""
                for idx, au in enumerate(author_list):
                    last = au.find("LastName")
                    fore = au.find("ForeName")
                    initials = au.find("Initials")

                    if last is None or not last.text:
                        continue
                    last_name = last.text.strip()
                    if idx == 0:
                        first_author_lastname = last_name

                    if fore is not None and fore.text:
                        first_name = fore.text.strip()
                    elif initials is not None and initials.text:
                        first_name = initials.text.strip()
                    else:
                        first_name = ""
                    if first_name:
                        authors.append(f"{last_name}, {first_name}")
                    else:
                        authors.append(last_name)

                authors_str = " and ".join(authors)

                # DOI
                doi = ""
                for aid in article.findall(".//ArticleIdList/ArticleId"):
                    if aid.get("IdType") == "doi" and aid.text:
                        doi = aid.text.strip()
                        break

                # BibTeX key
                key_author = first_author_lastname if first_author_lastname else "UnknownAuthor"
                key_year = year if year else "n.d."
                journal_abbrev = journal.replace(" ", "")[:20]
                bib_key = f"{key_author}{key_year}_{journal_abbrev}"
                bib_key = "".join(ch for ch in bib_key if ch.isalnum() or ch in "_.-")

                def esc(s):
                    return s.replace("{", "").replace("}", "")

                bib_lines = [
                    f"@article{{{bib_key},",
                    f"  author  = {{{esc(authors_str)}}},",
                    f"  title   = {{{esc(title)}}},",
                    f"  journal = {{{esc(journal)}}},",
                    f"  year    = {{{year}}},",
                    f"  volume  = {{{volume}}},",
                    f"  number  = {{{issue}}},",
                    f"  pages   = {{{pages}}},",
                    f"  doi     = {{{doi}}},",
                    f"  pmid    = {{{pmid}}}",
                    "}"
                ]
                bibtex_str = "\n".join(bib_lines)

                out[pmid] = {
                    "Title": title,
                    "Journal": journal,
                    "Year": year,
                    "FirstAuthor": first_author_lastname,
                    "DOI": doi,
                    "Volume": volume,
                    "Issue": issue,
                    "Pages": pages,
                    "BibTeX_Key": bib_key,
                    "BibTeX": bibtex_str,
                }

            except Exception as e:
                print(f"⚠ Error parsing article in EFetch: {e}")
                continue

        time.sleep(SLEEP_SECS)

    return out


def pick_earliest_pmid(pmids, details_dict):
    """
    Given a list of pmids and a details dict (pmid -> Year, etc.),
    return the PMID with the earliest publication year.
    If year is missing or not numeric, treat as very late (9999).
    """
    if not pmids:
        return None

    def year_as_int(pmid):
        y = details_dict.get(pmid, {}).get("Year", "")
        if isinstance(y, int):
            return y
        try:
            return int(y)
        except (TypeError, ValueError):
            return 9999

    return min(pmids, key=year_as_int)


# -------------------- MAIN LOGIC --------------------
def main():
    if not NCBI_EMAIL or NCBI_EMAIL == "your_email@domain.com":
        print("❌ Please set your real email address in NCBI_EMAIL.", file=sys.stderr)
        sys.exit(1)

    pathways = read_pathways(PATHWAY_FILE)
    print(f"📄 Loaded {len(pathways)} pathways from {PATHWAY_FILE}\n")

    out_rows = []

    for i, pathway in enumerate(pathways, 1):
        print(f"[{i}/{len(pathways)}] {pathway} → PubMed search (HIV/SIV + long-term ART context)")
        query = build_query(pathway)
        try:
            pmids = esearch_pmids(query, retmax=RETMAX)
            time.sleep(SLEEP_SECS)

            if not pmids:
                print("   ⚠ No PubMed hits.")
                out_rows.append({
                    "Pathway": pathway,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "No PubMed result in HIV/SIV + long-term ART context",
                })
                continue

            # Fetch metadata for all PMIDs, then pick earliest
            details = efetch_details(pmids)
            earliest_pmid = pick_earliest_pmid(pmids, details)

            if earliest_pmid is None or earliest_pmid not in details:
                print("   ⚠ Could not resolve earliest paper; writing NA row.")
                out_rows.append({
                    "Pathway": pathway,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "PMIDs found but metadata missing",
                })
                continue

            meta = details[earliest_pmid]
            print(f"   ✅ Earliest paper: PMID {earliest_pmid}, Year {meta.get('Year','')}")

            out_rows.append({
                "Pathway": pathway,
                "PMID": earliest_pmid,
                "Title": meta.get("Title", ""),
                "Journal": meta.get("Journal", ""),
                "Year": meta.get("Year", ""),
                "DOI": meta.get("DOI", ""),
                "PubMed_URL": pubmed_url(earliest_pmid),
                "BibTeX_Key": meta.get("BibTeX_Key", ""),
                "BibTeX": meta.get("BibTeX", ""),
                "Query": query,
                "Note": "",
            })

        except Exception as e:
            print(f"   ❌ Error for pathway '{pathway}': {e}")
            out_rows.append({
                "Pathway": pathway,
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "DOI": "NA",
                "PubMed_URL": "NA",
                "BibTeX_Key": "NA",
                "BibTeX": "NA",
                "Query": query,
                "Note": f"Error: {e}",
            })

        time.sleep(SLEEP_SECS)

    cols = [
        "Pathway", "PMID", "Title", "Journal", "Year",
        "DOI", "PubMed_URL", "BibTeX_Key", "BibTeX",
        "Query", "Note"
    ]

    # CSV
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            # sanitize quotes inside strings
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    # Excel
    df = pd.DataFrame(out_rows, columns=cols)
    df.to_excel(OUT_XLSX, index=False)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


# **ltart_baseline**

In [ ]:
#!/usr/bin/env python3
"""
Check whether canonical pathways are reported as affected under
LONG-TERM antiretroviral therapy (ART) in HIV/SIV literature.

Specifically, this helps you see if pathways are known to be altered in:
  - long-term ART–treated HIV/SIV-infected individuals
  - often compared to healthy / HIV-negative / uninfected controls.

Input:
    - A text file with one pathway term per line, e.g.:
        Interferon signaling
        Neuroinflammation signaling pathway
        VEGF signaling
      (no header; blank lines are ignored)

For each pathway:
    - Build a PubMed query:
        (pathway term in Title/Abstract)
        AND (HIV/SIV context terms)
        AND (long-term ART / chronic ART / virally suppressed on ART terms)
    - Retrieve PMIDs
    - If ≥1 hit:
        - Fetch detailed metadata via EFetch (XML)
        - Pick the *earliest* paper by publication year
        - Build a BibTeX entry
    - If no hits:
        - Write a row with NA fields

Outputs:
    - SIV_ltART_pathways_lit.csv
    - SIV_ltART_pathways_lit.xlsx

Columns:
    Pathway, PMID, Title, Journal, Year, DOI, PubMed_URL,
    BibTeX_Key, BibTeX, Query, Note
"""

import os
import time
import csv
import sys
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"   # REQUIRED by NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()  # optional but recommended

# <-- your input file for LT-ART pathways (one pathway per line)
PATHWAY_FILE = "/content/ltART_Baseline.txt"

OUT_CSV  = "SIV_ltART_Baseline_lit.csv"
OUT_XLSX = "SIV_ltART_Baseline_lit.xlsx"

RETMAX = 100                        # max PMIDs to retrieve per pathway
SLEEP_SECS = 0.35 if API_KEY else 0.6
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# HIV/SIV context – Title/Abstract and MeSH
HIV_TA_TERMS = [
    "HIV", "\"human immunodeficiency virus\"", "\"HIV-1\"", "\"HIV-2\"",
    "AIDS", "\"acquired immunodeficiency syndrome\"",
    "SIV", "\"simian immunodeficiency virus\"", "SHIV"
]

HIV_MESH_TERMS = [
    "HIV Infections[MeSH Terms]",
    "Acquired Immunodeficiency Syndrome[MeSH Terms]",
    "Simian Acquired Immunodeficiency Syndrome[MeSH Terms]"
]

# LONG-TERM ART / treatment context – Title/Abstract and MeSH
# (Includes generic ART terms + long-term / chronic / virally suppressed)
ART_TA_TERMS = [
    # long-term / chronic ART phrases
    "\"long-term antiretroviral therapy\"",
    "\"long term antiretroviral therapy\"",
    "\"long-term ART\"",
    "\"long term ART\"",
    "\"long-term HAART\"",
    "\"chronic HIV on ART\"",
    "\"chronically HIV-infected on ART\"",
    "\"chronically HIV infected on ART\"",
    "\"long-term cART\"",
    "\"long term cART\"",
    "\"prolonged ART\"",
    "\"sustained ART\"",
    "\"sustained virologic suppression\"",
    "\"sustained viral suppression\"",
    "\"virally suppressed on ART\"",
    "\"virally suppressed HIV\"",
    "\"ART-suppressed\"",
    "\"ART suppressed\"",
    "\"treated HIV infection\"",

    # keep generic ART terms as well
    "\"antiretroviral therapy\"",
    "\"antiretroviral treatment\"",
    "\"HAART\"",
    "\"highly active antiretroviral therapy\"",
    "\"cART\"",
    "\"combined antiretroviral therapy\"",
    "\"ART-treated\"",
    "\"ART treated\""
]

ART_MESH_TERMS = [
    "Anti-Retroviral Agents[MeSH Terms]",
    "Anti-HIV Agents[MeSH Terms]",
    "Antiretroviral Therapy, Highly Active[MeSH Terms]"
]

# Optional helper terms describing comparisons vs healthy controls.
# We will NOT hard-AND these (to avoid making queries too strict),
# but they are included in ART_TA_TERMS-like phrases above in practice.
HEALTHY_CONTROL_TA_TERMS = [
    "\"healthy controls\"",
    "\"HIV-negative\"",
    "\"HIV negative\"",
    "\"HIV-uninfected\"",
    "\"HIV uninfected\"",
    "\"seronegative controls\"",
    "\"uninfected controls\""
]


# -------------------- HELPERS --------------------
def read_pathways(path):
    """Read pathway names from a txt file (one per line)."""
    if not os.path.exists(path):
        print(f"❌ Pathway file not found: {path}", file=sys.stderr)
        sys.exit(1)
    pathways = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            term = line.strip()
            if term:
                pathways.append(term)
    return pathways


def build_query(pathway: str) -> str:
    """
    Build PubMed query:

      (pathway term in Title/Abstract)
      AND (HIV/SIV context in Title/Abstract or MeSH)
      AND (long-term / chronic ART context in Title/Abstract or MeSH)

    This tends to retrieve:
      - long-term ART-treated HIV/SIV cohorts,
      - often in comparison to healthy / HIV-negative controls.
    """
    t = pathway.replace('"', '')  # sanitize double quotes
    term_block = f"(\"{t}\"[Title/Abstract] OR {t}[Title/Abstract])"

    hiv_ta   = "(" + " OR ".join(HIV_TA_TERMS) + ")"
    hiv_mesh = "(" + " OR ".join(HIV_MESH_TERMS) + ")"
    hiv_any  = f"({hiv_ta} OR {hiv_mesh})"

    art_ta   = "(" + " OR ".join(ART_TA_TERMS) + ")"
    art_mesh = "(" + " OR ".join(ART_MESH_TERMS) + ")"
    art_any  = f"({art_ta} OR {art_mesh})"

    # Require BOTH HIV/SIV AND ART context
    q = f"{term_block} AND {hiv_any} AND {art_any}"

    # Optional: restrict to ART era (post-1996)
    # q = f'{q} AND ("1996"[Date - Publication] : "3000"[Date - Publication])'

    return q


def eutils_request(path: str, params: dict, retries: int = 3):
    """Robust E-utilities call with retries and basic rate limiting."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS_BASE}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ E-utilities request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_SECS * attempt)


def esearch_pmids(query: str, retmax: int = 50):
    """Search PubMed and return up to retmax PMIDs."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": retmax,
        "retmode": "json",
        "sort": "relevance",
    }
    resp = eutils_request("esearch.fcgi", params).json()
    return resp.get("esearchresult", {}).get("idlist", [])


def pubmed_url(pmid: str) -> str:
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


def efetch_details(pmids):
    """
    Use PubMed EFetch (XML) to get metadata and BibTeX-ready info for a list of PMIDs.
    Returns dict[pmid] = {
        "Title", "Journal", "Year", "FirstAuthor", "DOI",
        "Volume", "Issue", "Pages",
        "BibTeX_Key", "BibTeX"
    }
    """
    if not pmids:
        return {}

    out = {}
    CHUNK_SIZE = 200
    for i in range(0, len(pmids), CHUNK_SIZE):
        chunk = pmids[i:i + CHUNK_SIZE]
        params = {
            "db": "pubmed",
            "id": ",".join(chunk),
            "retmode": "xml",
        }
        r = eutils_request("efetch.fcgi", params)
        root = ET.fromstring(r.text)

        for article in root.findall(".//PubmedArticle"):
            try:
                pmid_elem = article.find(".//PMID")
                pmid = pmid_elem.text.strip() if pmid_elem is not None else None
                if not pmid:
                    continue

                art = article.find(".//Article")
                if art is None:
                    continue

                # Title
                title_elem = art.find("ArticleTitle")
                title = "".join(title_elem.itertext()).strip() if title_elem is not None else ""

                # Journal + year, volume, issue, pages
                journal_elem = art.find("Journal")
                journal_title_elem = journal_elem.find("Title") if journal_elem is not None else None
                journal = journal_title_elem.text.strip() if journal_title_elem is not None else ""

                pub_date = journal_elem.find("JournalIssue/PubDate") if journal_elem is not None else None
                year = ""
                if pub_date is not None:
                    y = pub_date.find("Year")
                    if y is not None and y.text and y.text.strip().isdigit():
                        year = y.text.strip()
                    else:
                        md = pub_date.find("MedlineDate")
                        if md is not None and md.text:
                            for tok in md.text.split():
                                if tok.isdigit() and len(tok) == 4:
                                    year = tok
                                    break

                medline_pg = art.find("Pagination/MedlinePgn")
                pages = medline_pg.text.strip() if medline_pg is not None else ""
                vol = art.find("JournalIssue/Volume")
                volume = vol.text.strip() if vol is not None else ""
                iss = art.find("JournalIssue/Issue")
                issue = iss.text.strip() if iss is not None else ""

                # Authors
                author_list = art.findall("AuthorList/Author")
                authors = []
                first_author_lastname = ""
                for idx, au in enumerate(author_list):
                    last = au.find("LastName")
                    fore = au.find("ForeName")
                    initials = au.find("Initials")

                    if last is None or not last.text:
                        continue
                    last_name = last.text.strip()
                    if idx == 0:
                        first_author_lastname = last_name

                    if fore is not None and fore.text:
                        first_name = fore.text.strip()
                    elif initials is not None and initials.text:
                        first_name = initials.text.strip()
                    else:
                        first_name = ""
                    if first_name:
                        authors.append(f"{last_name}, {first_name}")
                    else:
                        authors.append(last_name)

                authors_str = " and ".join(authors)

                # DOI
                doi = ""
                for aid in article.findall(".//ArticleIdList/ArticleId"):
                    if aid.get("IdType") == "doi" and aid.text:
                        doi = aid.text.strip()
                        break

                # BibTeX key
                key_author = first_author_lastname if first_author_lastname else "UnknownAuthor"
                key_year = year if year else "n.d."
                journal_abbrev = journal.replace(" ", "")[:20]
                bib_key = f"{key_author}{key_year}_{journal_abbrev}"
                bib_key = "".join(ch for ch in bib_key if ch.isalnum() or ch in "_.-")

                def esc(s):
                    return s.replace("{", "").replace("}", "")

                bib_lines = [
                    f"@article{{{bib_key},",
                    f"  author  = {{{esc(authors_str)}}},",
                    f"  title   = {{{esc(title)}}},",
                    f"  journal = {{{esc(journal)}}},",
                    f"  year    = {{{year}}},",
                    f"  volume  = {{{volume}}},",
                    f"  number  = {{{issue}}},",
                    f"  pages   = {{{pages}}},",
                    f"  doi     = {{{doi}}},",
                    f"  pmid    = {{{pmid}}}",
                    "}"
                ]
                bibtex_str = "\n".join(bib_lines)

                out[pmid] = {
                    "Title": title,
                    "Journal": journal,
                    "Year": year,
                    "FirstAuthor": first_author_lastname,
                    "DOI": doi,
                    "Volume": volume,
                    "Issue": issue,
                    "Pages": pages,
                    "BibTeX_Key": bib_key,
                    "BibTeX": bibtex_str,
                }

            except Exception as e:
                print(f"⚠ Error parsing article in EFetch: {e}")
                continue

        time.sleep(SLEEP_SECS)

    return out


def pick_earliest_pmid(pmids, details_dict):
    """
    Given a list of pmids and a details dict (pmid -> Year, etc.),
    return the PMID with the earliest publication year.
    If year is missing or not numeric, treat as very late (9999).
    """
    if not pmids:
        return None

    def year_as_int(pmid):
        y = details_dict.get(pmid, {}).get("Year", "")
        if isinstance(y, int):
            return y
        try:
            return int(y)
        except (TypeError, ValueError):
            return 9999

    return min(pmids, key=year_as_int)


# -------------------- MAIN LOGIC --------------------
def main():
    if not NCBI_EMAIL or NCBI_EMAIL == "your_email@domain.com":
        print("❌ Please set your real email address in NCBI_EMAIL.", file=sys.stderr)
        sys.exit(1)

    pathways = read_pathways(PATHWAY_FILE)
    print(f"📄 Loaded {len(pathways)} pathways from {PATHWAY_FILE}\n")

    out_rows = []

    for i, pathway in enumerate(pathways, 1):
        print(f"[{i}/{len(pathways)}] {pathway} → PubMed search (HIV/SIV + long-term ART context)")
        query = build_query(pathway)
        try:
            pmids = esearch_pmids(query, retmax=RETMAX)
            time.sleep(SLEEP_SECS)

            if not pmids:
                print("   ⚠ No PubMed hits.")
                out_rows.append({
                    "Pathway": pathway,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "No PubMed result in HIV/SIV + long-term ART context",
                })
                continue

            # Fetch metadata for all PMIDs, then pick earliest
            details = efetch_details(pmids)
            earliest_pmid = pick_earliest_pmid(pmids, details)

            if earliest_pmid is None or earliest_pmid not in details:
                print("   ⚠ Could not resolve earliest paper; writing NA row.")
                out_rows.append({
                    "Pathway": pathway,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "PMIDs found but metadata missing",
                })
                continue

            meta = details[earliest_pmid]
            print(f"   ✅ Earliest paper: PMID {earliest_pmid}, Year {meta.get('Year','')}")

            out_rows.append({
                "Pathway": pathway,
                "PMID": earliest_pmid,
                "Title": meta.get("Title", ""),
                "Journal": meta.get("Journal", ""),
                "Year": meta.get("Year", ""),
                "DOI": meta.get("DOI", ""),
                "PubMed_URL": pubmed_url(earliest_pmid),
                "BibTeX_Key": meta.get("BibTeX_Key", ""),
                "BibTeX": meta.get("BibTeX", ""),
                "Query": query,
                "Note": "",
            })

        except Exception as e:
            print(f"   ❌ Error for pathway '{pathway}': {e}")
            out_rows.append({
                "Pathway": pathway,
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "DOI": "NA",
                "PubMed_URL": "NA",
                "BibTeX_Key": "NA",
                "BibTeX": "NA",
                "Query": query,
                "Note": f"Error: {e}",
            })

        time.sleep(SLEEP_SECS)

    cols = [
        "Pathway", "PMID", "Title", "Journal", "Year",
        "DOI", "PubMed_URL", "BibTeX_Key", "BibTeX",
        "Query", "Note"
    ]

    # CSV
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            # sanitize quotes inside strings
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    # Excel
    df = pd.DataFrame(out_rows, columns=cols)
    df.to_excel(OUT_XLSX, index=False)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


📄 Loaded 30 pathways from /content/ltART_Baseline.txt

[1/30] canonical pathways → PubMed search (HIV/SIV + long-term ART context)
   ✅ Earliest paper: PMID 17266559, Year 2007
[2/30] Keratinization → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[3/30] Synaptogenesis Signaling Pathway → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[4/30] KEGG → PubMed search (HIV/SIV + long-term ART context)
   ✅ Earliest paper: PMID 25608975, Year 2015
[5/30] Endocytosis → PubMed search (HIV/SIV + long-term ART context)
   ✅ Earliest paper: PMID 9545196, Year 1998
[6/30] Synaptic vesicle cycle → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[7/30] Bladder cancer → PubMed search (HIV/SIV + long-term ART context)
   ✅ Earliest paper: PMID 16140456, Year 2005
[8/30] NOD-like receptor signaling pathway → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[9/30] Transcriptional misregulation in cancer → PubMed search (

# **ltart start**

In [ ]:
#!/usr/bin/env python3
"""
Check whether canonical pathways are reported as affected under
LONG-TERM antiretroviral therapy (ART) in HIV/SIV literature.

Specifically, this helps you see if pathways are known to be altered in:
  - long-term ART–treated HIV/SIV-infected individuals
  - often compared to healthy / HIV-negative / uninfected controls.

Input:
    - A text file with one pathway term per line, e.g.:
        Interferon signaling
        Neuroinflammation signaling pathway
        VEGF signaling
      (no header; blank lines are ignored)

For each pathway:
    - Build a PubMed query:
        (pathway term in Title/Abstract)
        AND (HIV/SIV context terms)
        AND (long-term ART / chronic ART / virally suppressed on ART terms)
    - Retrieve PMIDs
    - If ≥1 hit:
        - Fetch detailed metadata via EFetch (XML)
        - Pick the *earliest* paper by publication year
        - Build a BibTeX entry
    - If no hits:
        - Write a row with NA fields

Outputs:
    - SIV_ltART_pathways_lit.csv
    - SIV_ltART_pathways_lit.xlsx

Columns:
    Pathway, PMID, Title, Journal, Year, DOI, PubMed_URL,
    BibTeX_Key, BibTeX, Query, Note
"""

import os
import time
import csv
import sys
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"   # REQUIRED by NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()  # optional but recommended

# <-- your input file for LT-ART pathways (one pathway per line)
PATHWAY_FILE = "/content/ltART_stART.txt"

OUT_CSV  = "SIV_ltART_stART_lit.csv"
OUT_XLSX = "SIV_ltART_stART_lit.xlsx"

RETMAX = 100                        # max PMIDs to retrieve per pathway
SLEEP_SECS = 0.35 if API_KEY else 0.6
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# HIV/SIV context – Title/Abstract and MeSH
HIV_TA_TERMS = [
    "HIV", "\"human immunodeficiency virus\"", "\"HIV-1\"", "\"HIV-2\"",
    "AIDS", "\"acquired immunodeficiency syndrome\"",
    "SIV", "\"simian immunodeficiency virus\"", "SHIV"
]

HIV_MESH_TERMS = [
    "HIV Infections[MeSH Terms]",
    "Acquired Immunodeficiency Syndrome[MeSH Terms]",
    "Simian Acquired Immunodeficiency Syndrome[MeSH Terms]"
]

# LONG-TERM ART / treatment context – Title/Abstract and MeSH
# (Includes generic ART terms + long-term / chronic / virally suppressed)
ART_TA_TERMS = [
    # long-term / chronic ART phrases
    "\"long-term antiretroviral therapy\"",
    "\"long term antiretroviral therapy\"",
    "\"long-term ART\"",
    "\"long term ART\"",
    "\"long-term HAART\"",
    "\"chronic HIV on ART\"",
    "\"chronically HIV-infected on ART\"",
    "\"chronically HIV infected on ART\"",
    "\"long-term cART\"",
    "\"long term cART\"",
    "\"prolonged ART\"",
    "\"sustained ART\"",
    "\"sustained virologic suppression\"",
    "\"sustained viral suppression\"",
    "\"virally suppressed on ART\"",
    "\"virally suppressed HIV\"",
    "\"ART-suppressed\"",
    "\"ART suppressed\"",
    "\"treated HIV infection\"",

    # keep generic ART terms as well
    "\"antiretroviral therapy\"",
    "\"antiretroviral treatment\"",
    "\"HAART\"",
    "\"highly active antiretroviral therapy\"",
    "\"cART\"",
    "\"combined antiretroviral therapy\"",
    "\"ART-treated\"",
    "\"ART treated\""
]

ART_MESH_TERMS = [
    "Anti-Retroviral Agents[MeSH Terms]",
    "Anti-HIV Agents[MeSH Terms]",
    "Antiretroviral Therapy, Highly Active[MeSH Terms]"
]

# Optional helper terms describing comparisons vs healthy controls.
# We will NOT hard-AND these (to avoid making queries too strict),
# but they are included in ART_TA_TERMS-like phrases above in practice.
HEALTHY_CONTROL_TA_TERMS = [
    "\"healthy controls\"",
    "\"HIV-negative\"",
    "\"HIV negative\"",
    "\"HIV-uninfected\"",
    "\"HIV uninfected\"",
    "\"seronegative controls\"",
    "\"uninfected controls\""
]


# -------------------- HELPERS --------------------
def read_pathways(path):
    """Read pathway names from a txt file (one per line)."""
    if not os.path.exists(path):
        print(f"❌ Pathway file not found: {path}", file=sys.stderr)
        sys.exit(1)
    pathways = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            term = line.strip()
            if term:
                pathways.append(term)
    return pathways


def build_query(pathway: str) -> str:
    """
    Build PubMed query:

      (pathway term in Title/Abstract)
      AND (HIV/SIV context in Title/Abstract or MeSH)
      AND (long-term / chronic ART context in Title/Abstract or MeSH)

    This tends to retrieve:
      - long-term ART-treated HIV/SIV cohorts,
      - often in comparison to healthy / HIV-negative controls.
    """
    t = pathway.replace('"', '')  # sanitize double quotes
    term_block = f"(\"{t}\"[Title/Abstract] OR {t}[Title/Abstract])"

    hiv_ta   = "(" + " OR ".join(HIV_TA_TERMS) + ")"
    hiv_mesh = "(" + " OR ".join(HIV_MESH_TERMS) + ")"
    hiv_any  = f"({hiv_ta} OR {hiv_mesh})"

    art_ta   = "(" + " OR ".join(ART_TA_TERMS) + ")"
    art_mesh = "(" + " OR ".join(ART_MESH_TERMS) + ")"
    art_any  = f"({art_ta} OR {art_mesh})"

    # Require BOTH HIV/SIV AND ART context
    q = f"{term_block} AND {hiv_any} AND {art_any}"

    # Optional: restrict to ART era (post-1996)
    # q = f'{q} AND ("1996"[Date - Publication] : "3000"[Date - Publication])'

    return q


def eutils_request(path: str, params: dict, retries: int = 3):
    """Robust E-utilities call with retries and basic rate limiting."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS_BASE}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ E-utilities request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_SECS * attempt)


def esearch_pmids(query: str, retmax: int = 50):
    """Search PubMed and return up to retmax PMIDs."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": retmax,
        "retmode": "json",
        "sort": "relevance",
    }
    resp = eutils_request("esearch.fcgi", params).json()
    return resp.get("esearchresult", {}).get("idlist", [])


def pubmed_url(pmid: str) -> str:
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


def efetch_details(pmids):
    """
    Use PubMed EFetch (XML) to get metadata and BibTeX-ready info for a list of PMIDs.
    Returns dict[pmid] = {
        "Title", "Journal", "Year", "FirstAuthor", "DOI",
        "Volume", "Issue", "Pages",
        "BibTeX_Key", "BibTeX"
    }
    """
    if not pmids:
        return {}

    out = {}
    CHUNK_SIZE = 200
    for i in range(0, len(pmids), CHUNK_SIZE):
        chunk = pmids[i:i + CHUNK_SIZE]
        params = {
            "db": "pubmed",
            "id": ",".join(chunk),
            "retmode": "xml",
        }
        r = eutils_request("efetch.fcgi", params)
        root = ET.fromstring(r.text)

        for article in root.findall(".//PubmedArticle"):
            try:
                pmid_elem = article.find(".//PMID")
                pmid = pmid_elem.text.strip() if pmid_elem is not None else None
                if not pmid:
                    continue

                art = article.find(".//Article")
                if art is None:
                    continue

                # Title
                title_elem = art.find("ArticleTitle")
                title = "".join(title_elem.itertext()).strip() if title_elem is not None else ""

                # Journal + year, volume, issue, pages
                journal_elem = art.find("Journal")
                journal_title_elem = journal_elem.find("Title") if journal_elem is not None else None
                journal = journal_title_elem.text.strip() if journal_title_elem is not None else ""

                pub_date = journal_elem.find("JournalIssue/PubDate") if journal_elem is not None else None
                year = ""
                if pub_date is not None:
                    y = pub_date.find("Year")
                    if y is not None and y.text and y.text.strip().isdigit():
                        year = y.text.strip()
                    else:
                        md = pub_date.find("MedlineDate")
                        if md is not None and md.text:
                            for tok in md.text.split():
                                if tok.isdigit() and len(tok) == 4:
                                    year = tok
                                    break

                medline_pg = art.find("Pagination/MedlinePgn")
                pages = medline_pg.text.strip() if medline_pg is not None else ""
                vol = art.find("JournalIssue/Volume")
                volume = vol.text.strip() if vol is not None else ""
                iss = art.find("JournalIssue/Issue")
                issue = iss.text.strip() if iss is not None else ""

                # Authors
                author_list = art.findall("AuthorList/Author")
                authors = []
                first_author_lastname = ""
                for idx, au in enumerate(author_list):
                    last = au.find("LastName")
                    fore = au.find("ForeName")
                    initials = au.find("Initials")

                    if last is None or not last.text:
                        continue
                    last_name = last.text.strip()
                    if idx == 0:
                        first_author_lastname = last_name

                    if fore is not None and fore.text:
                        first_name = fore.text.strip()
                    elif initials is not None and initials.text:
                        first_name = initials.text.strip()
                    else:
                        first_name = ""
                    if first_name:
                        authors.append(f"{last_name}, {first_name}")
                    else:
                        authors.append(last_name)

                authors_str = " and ".join(authors)

                # DOI
                doi = ""
                for aid in article.findall(".//ArticleIdList/ArticleId"):
                    if aid.get("IdType") == "doi" and aid.text:
                        doi = aid.text.strip()
                        break

                # BibTeX key
                key_author = first_author_lastname if first_author_lastname else "UnknownAuthor"
                key_year = year if year else "n.d."
                journal_abbrev = journal.replace(" ", "")[:20]
                bib_key = f"{key_author}{key_year}_{journal_abbrev}"
                bib_key = "".join(ch for ch in bib_key if ch.isalnum() or ch in "_.-")

                def esc(s):
                    return s.replace("{", "").replace("}", "")

                bib_lines = [
                    f"@article{{{bib_key},",
                    f"  author  = {{{esc(authors_str)}}},",
                    f"  title   = {{{esc(title)}}},",
                    f"  journal = {{{esc(journal)}}},",
                    f"  year    = {{{year}}},",
                    f"  volume  = {{{volume}}},",
                    f"  number  = {{{issue}}},",
                    f"  pages   = {{{pages}}},",
                    f"  doi     = {{{doi}}},",
                    f"  pmid    = {{{pmid}}}",
                    "}"
                ]
                bibtex_str = "\n".join(bib_lines)

                out[pmid] = {
                    "Title": title,
                    "Journal": journal,
                    "Year": year,
                    "FirstAuthor": first_author_lastname,
                    "DOI": doi,
                    "Volume": volume,
                    "Issue": issue,
                    "Pages": pages,
                    "BibTeX_Key": bib_key,
                    "BibTeX": bibtex_str,
                }

            except Exception as e:
                print(f"⚠ Error parsing article in EFetch: {e}")
                continue

        time.sleep(SLEEP_SECS)

    return out


def pick_earliest_pmid(pmids, details_dict):
    """
    Given a list of pmids and a details dict (pmid -> Year, etc.),
    return the PMID with the earliest publication year.
    If year is missing or not numeric, treat as very late (9999).
    """
    if not pmids:
        return None

    def year_as_int(pmid):
        y = details_dict.get(pmid, {}).get("Year", "")
        if isinstance(y, int):
            return y
        try:
            return int(y)
        except (TypeError, ValueError):
            return 9999

    return min(pmids, key=year_as_int)


# -------------------- MAIN LOGIC --------------------
def main():
    if not NCBI_EMAIL or NCBI_EMAIL == "your_email@domain.com":
        print("❌ Please set your real email address in NCBI_EMAIL.", file=sys.stderr)
        sys.exit(1)

    pathways = read_pathways(PATHWAY_FILE)
    print(f"📄 Loaded {len(pathways)} pathways from {PATHWAY_FILE}\n")

    out_rows = []

    for i, pathway in enumerate(pathways, 1):
        print(f"[{i}/{len(pathways)}] {pathway} → PubMed search (HIV/SIV + long-term ART context)")
        query = build_query(pathway)
        try:
            pmids = esearch_pmids(query, retmax=RETMAX)
            time.sleep(SLEEP_SECS)

            if not pmids:
                print("   ⚠ No PubMed hits.")
                out_rows.append({
                    "Pathway": pathway,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "No PubMed result in HIV/SIV + long-term ART context",
                })
                continue

            # Fetch metadata for all PMIDs, then pick earliest
            details = efetch_details(pmids)
            earliest_pmid = pick_earliest_pmid(pmids, details)

            if earliest_pmid is None or earliest_pmid not in details:
                print("   ⚠ Could not resolve earliest paper; writing NA row.")
                out_rows.append({
                    "Pathway": pathway,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "PMIDs found but metadata missing",
                })
                continue

            meta = details[earliest_pmid]
            print(f"   ✅ Earliest paper: PMID {earliest_pmid}, Year {meta.get('Year','')}")

            out_rows.append({
                "Pathway": pathway,
                "PMID": earliest_pmid,
                "Title": meta.get("Title", ""),
                "Journal": meta.get("Journal", ""),
                "Year": meta.get("Year", ""),
                "DOI": meta.get("DOI", ""),
                "PubMed_URL": pubmed_url(earliest_pmid),
                "BibTeX_Key": meta.get("BibTeX_Key", ""),
                "BibTeX": meta.get("BibTeX", ""),
                "Query": query,
                "Note": "",
            })

        except Exception as e:
            print(f"   ❌ Error for pathway '{pathway}': {e}")
            out_rows.append({
                "Pathway": pathway,
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "DOI": "NA",
                "PubMed_URL": "NA",
                "BibTeX_Key": "NA",
                "BibTeX": "NA",
                "Query": query,
                "Note": f"Error: {e}",
            })

        time.sleep(SLEEP_SECS)

    cols = [
        "Pathway", "PMID", "Title", "Journal", "Year",
        "DOI", "PubMed_URL", "BibTeX_Key", "BibTeX",
        "Query", "Note"
    ]

    # CSV
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            # sanitize quotes inside strings
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    # Excel
    df = pd.DataFrame(out_rows, columns=cols)
    df.to_excel(OUT_XLSX, index=False)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


📄 Loaded 63 pathways from /content/ltART_stART.txt

[1/63] ltART_stART CP → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[2/63] S100 Family Signaling Pathway → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[3/63] EPH-Ephrin signaling → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[4/63] Neutrophil degranulation → PubMed search (HIV/SIV + long-term ART context)
   ✅ Earliest paper: PMID 39066199, Year 2024
[5/63] NF1 RAS Signaling Pathway → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[6/63] Breast Cancer Regulation by Stathmin1 → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[7/63] CREB Signaling in Neurons → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[8/63] Molecular Mechanisms of Cancer → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[9/63] Zn Homeostasis Signaling Pathway → PubMed search (HIV/SIV + long-term ART contex

# **ltart preart**

In [ ]:
#!/usr/bin/env python3
"""
Check whether canonical pathways are reported as affected under
LONG-TERM antiretroviral therapy (ART) in HIV/SIV literature.

Specifically, this helps you see if pathways are known to be altered in:
  - long-term ART–treated HIV/SIV-infected individuals
  - often compared to healthy / HIV-negative / uninfected controls.

Input:
    - A text file with one pathway term per line, e.g.:
        Interferon signaling
        Neuroinflammation signaling pathway
        VEGF signaling
      (no header; blank lines are ignored)

For each pathway:
    - Build a PubMed query:
        (pathway term in Title/Abstract)
        AND (HIV/SIV context terms)
        AND (long-term ART / chronic ART / virally suppressed on ART terms)
    - Retrieve PMIDs
    - If ≥1 hit:
        - Fetch detailed metadata via EFetch (XML)
        - Pick the *earliest* paper by publication year
        - Build a BibTeX entry
    - If no hits:
        - Write a row with NA fields

Outputs:
    - SIV_ltART_pathways_lit.csv
    - SIV_ltART_pathways_lit.xlsx

Columns:
    Pathway, PMID, Title, Journal, Year, DOI, PubMed_URL,
    BibTeX_Key, BibTeX, Query, Note
"""

import os
import time
import csv
import sys
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"   # REQUIRED by NCBI
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()  # optional but recommended

# <-- your input file for LT-ART pathways (one pathway per line)
PATHWAY_FILE = "ltART_preART.txt"

OUT_CSV  = "SIV_ltART_pathways_lit.csv"
OUT_XLSX = "SIV_ltART_pathways_lit.xlsx"

RETMAX = 100                        # max PMIDs to retrieve per pathway
SLEEP_SECS = 0.35 if API_KEY else 0.6
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# HIV/SIV context – Title/Abstract and MeSH
HIV_TA_TERMS = [
    "HIV", "\"human immunodeficiency virus\"", "\"HIV-1\"", "\"HIV-2\"",
    "AIDS", "\"acquired immunodeficiency syndrome\"",
    "SIV", "\"simian immunodeficiency virus\"", "SHIV"
]

HIV_MESH_TERMS = [
    "HIV Infections[MeSH Terms]",
    "Acquired Immunodeficiency Syndrome[MeSH Terms]",
    "Simian Acquired Immunodeficiency Syndrome[MeSH Terms]"
]

# LONG-TERM ART / treatment context – Title/Abstract and MeSH
# (Includes generic ART terms + long-term / chronic / virally suppressed)
ART_TA_TERMS = [
    # long-term / chronic ART phrases
    "\"long-term antiretroviral therapy\"",
    "\"long term antiretroviral therapy\"",
    "\"long-term ART\"",
    "\"long term ART\"",
    "\"long-term HAART\"",
    "\"chronic HIV on ART\"",
    "\"chronically HIV-infected on ART\"",
    "\"chronically HIV infected on ART\"",
    "\"long-term cART\"",
    "\"long term cART\"",
    "\"prolonged ART\"",
    "\"sustained ART\"",
    "\"sustained virologic suppression\"",
    "\"sustained viral suppression\"",
    "\"virally suppressed on ART\"",
    "\"virally suppressed HIV\"",
    "\"ART-suppressed\"",
    "\"ART suppressed\"",
    "\"treated HIV infection\"",

    # keep generic ART terms as well
    "\"antiretroviral therapy\"",
    "\"antiretroviral treatment\"",
    "\"HAART\"",
    "\"highly active antiretroviral therapy\"",
    "\"cART\"",
    "\"combined antiretroviral therapy\"",
    "\"ART-treated\"",
    "\"ART treated\""
]

ART_MESH_TERMS = [
    "Anti-Retroviral Agents[MeSH Terms]",
    "Anti-HIV Agents[MeSH Terms]",
    "Antiretroviral Therapy, Highly Active[MeSH Terms]"
]

# Optional helper terms describing comparisons vs healthy controls.
# We will NOT hard-AND these (to avoid making queries too strict),
# but they are included in ART_TA_TERMS-like phrases above in practice.
HEALTHY_CONTROL_TA_TERMS = [
    "\"healthy controls\"",
    "\"HIV-negative\"",
    "\"HIV negative\"",
    "\"HIV-uninfected\"",
    "\"HIV uninfected\"",
    "\"seronegative controls\"",
    "\"uninfected controls\""
]


# -------------------- HELPERS --------------------
def read_pathways(path):
    """Read pathway names from a txt file (one per line)."""
    if not os.path.exists(path):
        print(f"❌ Pathway file not found: {path}", file=sys.stderr)
        sys.exit(1)
    pathways = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            term = line.strip()
            if term:
                pathways.append(term)
    return pathways


def build_query(pathway: str) -> str:
    """
    Build PubMed query:

      (pathway term in Title/Abstract)
      AND (HIV/SIV context in Title/Abstract or MeSH)
      AND (long-term / chronic ART context in Title/Abstract or MeSH)

    This tends to retrieve:
      - long-term ART-treated HIV/SIV cohorts,
      - often in comparison to healthy / HIV-negative controls.
    """
    t = pathway.replace('"', '')  # sanitize double quotes
    term_block = f"(\"{t}\"[Title/Abstract] OR {t}[Title/Abstract])"

    hiv_ta   = "(" + " OR ".join(HIV_TA_TERMS) + ")"
    hiv_mesh = "(" + " OR ".join(HIV_MESH_TERMS) + ")"
    hiv_any  = f"({hiv_ta} OR {hiv_mesh})"

    art_ta   = "(" + " OR ".join(ART_TA_TERMS) + ")"
    art_mesh = "(" + " OR ".join(ART_MESH_TERMS) + ")"
    art_any  = f"({art_ta} OR {art_mesh})"

    # Require BOTH HIV/SIV AND ART context
    q = f"{term_block} AND {hiv_any} AND {art_any}"

    # Optional: restrict to ART era (post-1996)
    # q = f'{q} AND ("1996"[Date - Publication] : "3000"[Date - Publication])'

    return q


def eutils_request(path: str, params: dict, retries: int = 3):
    """Robust E-utilities call with retries and basic rate limiting."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY
    url = f"{EUTILS_BASE}/{path}"
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            if attempt == retries:
                print(f"❌ E-utilities request failed after {retries} attempts: {e}")
                raise
            time.sleep(SLEEP_SECS * attempt)


def esearch_pmids(query: str, retmax: int = 50):
    """Search PubMed and return up to retmax PMIDs."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": retmax,
        "retmode": "json",
        "sort": "relevance",
    }
    resp = eutils_request("esearch.fcgi", params).json()
    return resp.get("esearchresult", {}).get("idlist", [])


def pubmed_url(pmid: str) -> str:
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


def efetch_details(pmids):
    """
    Use PubMed EFetch (XML) to get metadata and BibTeX-ready info for a list of PMIDs.
    Returns dict[pmid] = {
        "Title", "Journal", "Year", "FirstAuthor", "DOI",
        "Volume", "Issue", "Pages",
        "BibTeX_Key", "BibTeX"
    }
    """
    if not pmids:
        return {}

    out = {}
    CHUNK_SIZE = 200
    for i in range(0, len(pmids), CHUNK_SIZE):
        chunk = pmids[i:i + CHUNK_SIZE]
        params = {
            "db": "pubmed",
            "id": ",".join(chunk),
            "retmode": "xml",
        }
        r = eutils_request("efetch.fcgi", params)
        root = ET.fromstring(r.text)

        for article in root.findall(".//PubmedArticle"):
            try:
                pmid_elem = article.find(".//PMID")
                pmid = pmid_elem.text.strip() if pmid_elem is not None else None
                if not pmid:
                    continue

                art = article.find(".//Article")
                if art is None:
                    continue

                # Title
                title_elem = art.find("ArticleTitle")
                title = "".join(title_elem.itertext()).strip() if title_elem is not None else ""

                # Journal + year, volume, issue, pages
                journal_elem = art.find("Journal")
                journal_title_elem = journal_elem.find("Title") if journal_elem is not None else None
                journal = journal_title_elem.text.strip() if journal_title_elem is not None else ""

                pub_date = journal_elem.find("JournalIssue/PubDate") if journal_elem is not None else None
                year = ""
                if pub_date is not None:
                    y = pub_date.find("Year")
                    if y is not None and y.text and y.text.strip().isdigit():
                        year = y.text.strip()
                    else:
                        md = pub_date.find("MedlineDate")
                        if md is not None and md.text:
                            for tok in md.text.split():
                                if tok.isdigit() and len(tok) == 4:
                                    year = tok
                                    break

                medline_pg = art.find("Pagination/MedlinePgn")
                pages = medline_pg.text.strip() if medline_pg is not None else ""
                vol = art.find("JournalIssue/Volume")
                volume = vol.text.strip() if vol is not None else ""
                iss = art.find("JournalIssue/Issue")
                issue = iss.text.strip() if iss is not None else ""

                # Authors
                author_list = art.findall("AuthorList/Author")
                authors = []
                first_author_lastname = ""
                for idx, au in enumerate(author_list):
                    last = au.find("LastName")
                    fore = au.find("ForeName")
                    initials = au.find("Initials")

                    if last is None or not last.text:
                        continue
                    last_name = last.text.strip()
                    if idx == 0:
                        first_author_lastname = last_name

                    if fore is not None and fore.text:
                        first_name = fore.text.strip()
                    elif initials is not None and initials.text:
                        first_name = initials.text.strip()
                    else:
                        first_name = ""
                    if first_name:
                        authors.append(f"{last_name}, {first_name}")
                    else:
                        authors.append(last_name)

                authors_str = " and ".join(authors)

                # DOI
                doi = ""
                for aid in article.findall(".//ArticleIdList/ArticleId"):
                    if aid.get("IdType") == "doi" and aid.text:
                        doi = aid.text.strip()
                        break

                # BibTeX key
                key_author = first_author_lastname if first_author_lastname else "UnknownAuthor"
                key_year = year if year else "n.d."
                journal_abbrev = journal.replace(" ", "")[:20]
                bib_key = f"{key_author}{key_year}_{journal_abbrev}"
                bib_key = "".join(ch for ch in bib_key if ch.isalnum() or ch in "_.-")

                def esc(s):
                    return s.replace("{", "").replace("}", "")

                bib_lines = [
                    f"@article{{{bib_key},",
                    f"  author  = {{{esc(authors_str)}}},",
                    f"  title   = {{{esc(title)}}},",
                    f"  journal = {{{esc(journal)}}},",
                    f"  year    = {{{year}}},",
                    f"  volume  = {{{volume}}},",
                    f"  number  = {{{issue}}},",
                    f"  pages   = {{{pages}}},",
                    f"  doi     = {{{doi}}},",
                    f"  pmid    = {{{pmid}}}",
                    "}"
                ]
                bibtex_str = "\n".join(bib_lines)

                out[pmid] = {
                    "Title": title,
                    "Journal": journal,
                    "Year": year,
                    "FirstAuthor": first_author_lastname,
                    "DOI": doi,
                    "Volume": volume,
                    "Issue": issue,
                    "Pages": pages,
                    "BibTeX_Key": bib_key,
                    "BibTeX": bibtex_str,
                }

            except Exception as e:
                print(f"⚠ Error parsing article in EFetch: {e}")
                continue

        time.sleep(SLEEP_SECS)

    return out


def pick_earliest_pmid(pmids, details_dict):
    """
    Given a list of pmids and a details dict (pmid -> Year, etc.),
    return the PMID with the earliest publication year.
    If year is missing or not numeric, treat as very late (9999).
    """
    if not pmids:
        return None

    def year_as_int(pmid):
        y = details_dict.get(pmid, {}).get("Year", "")
        if isinstance(y, int):
            return y
        try:
            return int(y)
        except (TypeError, ValueError):
            return 9999

    return min(pmids, key=year_as_int)


# -------------------- MAIN LOGIC --------------------
def main():
    if not NCBI_EMAIL or NCBI_EMAIL == "your_email@domain.com":
        print("❌ Please set your real email address in NCBI_EMAIL.", file=sys.stderr)
        sys.exit(1)

    pathways = read_pathways(PATHWAY_FILE)
    print(f"📄 Loaded {len(pathways)} pathways from {PATHWAY_FILE}\n")

    out_rows = []

    for i, pathway in enumerate(pathways, 1):
        print(f"[{i}/{len(pathways)}] {pathway} → PubMed search (HIV/SIV + long-term ART context)")
        query = build_query(pathway)
        try:
            pmids = esearch_pmids(query, retmax=RETMAX)
            time.sleep(SLEEP_SECS)

            if not pmids:
                print("   ⚠ No PubMed hits.")
                out_rows.append({
                    "Pathway": pathway,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "No PubMed result in HIV/SIV + long-term ART context",
                })
                continue

            # Fetch metadata for all PMIDs, then pick earliest
            details = efetch_details(pmids)
            earliest_pmid = pick_earliest_pmid(pmids, details)

            if earliest_pmid is None or earliest_pmid not in details:
                print("   ⚠ Could not resolve earliest paper; writing NA row.")
                out_rows.append({
                    "Pathway": pathway,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "PMIDs found but metadata missing",
                })
                continue

            meta = details[earliest_pmid]
            print(f"   ✅ Earliest paper: PMID {earliest_pmid}, Year {meta.get('Year','')}")

            out_rows.append({
                "Pathway": pathway,
                "PMID": earliest_pmid,
                "Title": meta.get("Title", ""),
                "Journal": meta.get("Journal", ""),
                "Year": meta.get("Year", ""),
                "DOI": meta.get("DOI", ""),
                "PubMed_URL": pubmed_url(earliest_pmid),
                "BibTeX_Key": meta.get("BibTeX_Key", ""),
                "BibTeX": meta.get("BibTeX", ""),
                "Query": query,
                "Note": "",
            })

        except Exception as e:
            print(f"   ❌ Error for pathway '{pathway}': {e}")
            out_rows.append({
                "Pathway": pathway,
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "DOI": "NA",
                "PubMed_URL": "NA",
                "BibTeX_Key": "NA",
                "BibTeX": "NA",
                "Query": query,
                "Note": f"Error: {e}",
            })

        time.sleep(SLEEP_SECS)

    cols = [
        "Pathway", "PMID", "Title", "Journal", "Year",
        "DOI", "PubMed_URL", "BibTeX_Key", "BibTeX",
        "Query", "Note"
    ]

    # CSV
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            # sanitize quotes inside strings
            safe_row = {
                k: (str(v).replace('"', "'") if isinstance(v, str) else v)
                for k, v in row.items()
            }
            w.writerow(safe_row)

    # Excel
    df = pd.DataFrame(out_rows, columns=cols)
    df.to_excel(OUT_XLSX, index=False)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")


if __name__ == "__main__":
    main()


📄 Loaded 33 pathways from ltART_preART.txt

[1/33] Ingenuity Canonical Pathways → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[2/33] Neurovascular Coupling Signaling Pathway → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[3/33] Signaling by TGF-beta Receptor Complex → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[4/33] Signaling by VEGF → PubMed search (HIV/SIV + long-term ART context)
   ✅ Earliest paper: PMID 17132220, Year 2006
[5/33] Virus Entry via Endocytic Pathways → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[6/33] Glioblastoma Multiforme Signaling → PubMed search (HIV/SIV + long-term ART context)
   ✅ Earliest paper: PMID 16033774, Year 2005
[7/33] ISG15 antiviral mechanism → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[8/33] Interferon alpha/beta signaling → PubMed search (HIV/SIV + long-term ART context)
   ⚠ No PubMed hits.
[9/33] Diseases or Functions

In [ ]:
#!/usr/bin/env python3
"""
PubMed Ontology/Controlled Vocabulary Literature Finder

Given a list of ontology names (one per line), for each term this script:
  - Builds a PubMed query that requires:
      (ontology term in Title/Abstract)
      AND (ontology / controlled vocabulary / terminology context via TA+MeSH)
    Optionally:
      AND (HIV/SIV context) if USE_DOMAIN_CONTEXT = True
  - Retrieves PMIDs (up to RETMAX)
  - Fetches metadata via EFetch (XML)
  - Picks the *earliest* paper by publication year
  - Creates a BibTeX entry
  - Outputs CSV + XLSX

Input file example (ontologies.txt):
  Gene Ontology
  Human Phenotype Ontology
  Disease Ontology
  UMLS
  Medical Subject Headings
  SNOMED CT
"""

import os
import time
import csv
import sys
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# -------------------- CONFIG --------------------
NCBI_EMAIL = "raysona07@gmail.com"  # REQUIRED by NCBI; set your real email
API_KEY = os.environ.get("NCBI_API_KEY", "").strip()  # optional but recommended

# Input: one ontology per line
ONTOLOGY_FILE = "/content/start_preart.txt"

# Output files
OUT_CSV  = "pubmed_ontologies_SP.csv"
OUT_XLSX = "pubmed_ontologies_SP.xlsx"

RETMAX = 200  # max PMIDs per ontology to retrieve
SLEEP_SECS = 0.35 if API_KEY else 0.6
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

# Toggle: restrict results to HIV/SIV space?
USE_DOMAIN_CONTEXT = False  # set True if you want HIV/SIV-only ontology usage

# -------------------- QUERY TERM BLOCKS --------------------
# Ontology / controlled vocabulary context (Title/Abstract)
ONTOLOGY_TA_TERMS = [
    "ontology[Title/Abstract]",
    "ontologies[Title/Abstract]",
    "\"controlled vocabulary\"[Title/Abstract]",
    "thesaurus[Title/Abstract]",
    "terminology[Title/Abstract]",
    "\"semantic\"[Title/Abstract]",
    "\"knowledge representation\"[Title/Abstract]",
    "\"annotation\"[Title/Abstract]"
]

# Ontology / controlled vocabulary context (MeSH)
ONTOLOGY_MESH_TERMS = [
    "\"Terminology as Topic\"[MeSH Terms]",
    "\"Vocabulary, Controlled\"[MeSH Terms]",
    "\"Medical Subject Headings\"[MeSH Terms]"
]

# Optional domain context (HIV/SIV) – Title/Abstract + MeSH
HIV_TA_TERMS = [
    "HIV[Title/Abstract]",
    "\"human immunodeficiency virus\"[Title/Abstract]",
    "\"HIV-1\"[Title/Abstract]",
    "\"HIV-2\"[Title/Abstract]",
    "AIDS[Title/Abstract]",
    "\"acquired immunodeficiency syndrome\"[Title/Abstract]",
    "SIV[Title/Abstract]",
    "\"simian immunodeficiency virus\"[Title/Abstract]",
    "SHIV[Title/Abstract]"
]

HIV_MESH_TERMS = [
    "HIV Infections[MeSH Terms]",
    "Acquired Immunodeficiency Syndrome[MeSH Terms]",
    "Simian Acquired Immunodeficiency Syndrome[MeSH Terms]"
]

# -------------------- HELPERS --------------------
def read_terms(path: str):
    """Read terms (one per line) from a txt file."""
    if not os.path.exists(path):
        print(f"❌ Input file not found: {path}", file=sys.stderr)
        sys.exit(1)
    terms = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            t = line.strip()
            if t:
                terms.append(t)
    return terms


def eutils_request(endpoint: str, params: dict, retries: int = 3):
    """Robust E-utilities call with retries + basic rate limiting."""
    params = params.copy()
    params["email"] = NCBI_EMAIL
    if API_KEY:
        params["api_key"] = API_KEY

    url = f"{EUTILS_BASE}/{endpoint}"
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as e:
            last_err = e
            time.sleep(SLEEP_SECS * attempt)

    raise RuntimeError(f"E-utilities request failed after {retries} attempts: {last_err}")


def build_query(ontology_term: str) -> str:
    """
    Query:
      (ontology term in Title/Abstract)
      AND (ontology/controlled vocabulary context via TA or MeSH)
      [optional] AND (HIV/SIV context via TA or MeSH)
    """
    t = ontology_term.replace('"', "").strip()
    term_block = f"(\"{t}\"[Title/Abstract] OR {t}[Title/Abstract])"

    onto_ta = "(" + " OR ".join(ONTOLOGY_TA_TERMS) + ")"
    onto_mesh = "(" + " OR ".join(ONTOLOGY_MESH_TERMS) + ")"
    onto_any = f"({onto_ta} OR {onto_mesh})"

    q = f"{term_block} AND {onto_any}"

    if USE_DOMAIN_CONTEXT:
        hiv_ta = "(" + " OR ".join(HIV_TA_TERMS) + ")"
        hiv_mesh = "(" + " OR ".join(HIV_MESH_TERMS) + ")"
        hiv_any = f"({hiv_ta} OR {hiv_mesh})"
        q = f"{q} AND {hiv_any}"

    return q


def esearch_pmids(query: str, retmax: int = 200):
    """Search PubMed and return up to retmax PMIDs."""
    params = {
        "db": "pubmed",
        "term": query,
        "retmax": retmax,
        "retmode": "json",
        "sort": "relevance",
    }
    resp = eutils_request("esearch.fcgi", params).json()
    return resp.get("esearchresult", {}).get("idlist", [])


def pubmed_url(pmid: str) -> str:
    return f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""


def efetch_details(pmids):
    """
    Fetch PubMed details via EFetch XML for a list of PMIDs.
    Returns dict[pmid] = metadata + bibtex
    """
    if not pmids:
        return {}

    out = {}
    CHUNK_SIZE = 200

    for i in range(0, len(pmids), CHUNK_SIZE):
        chunk = pmids[i:i + CHUNK_SIZE]
        params = {
            "db": "pubmed",
            "id": ",".join(chunk),
            "retmode": "xml",
        }
        r = eutils_request("efetch.fcgi", params)
        root = ET.fromstring(r.text)

        for article in root.findall(".//PubmedArticle"):
            try:
                pmid_elem = article.find(".//PMID")
                pmid = pmid_elem.text.strip() if pmid_elem is not None and pmid_elem.text else None
                if not pmid:
                    continue

                art = article.find(".//Article")
                if art is None:
                    continue

                # Title
                title_elem = art.find("ArticleTitle")
                title = "".join(title_elem.itertext()).strip() if title_elem is not None else ""

                # Journal
                journal_elem = art.find("Journal")
                journal_title_elem = journal_elem.find("Title") if journal_elem is not None else None
                journal = journal_title_elem.text.strip() if journal_title_elem is not None and journal_title_elem.text else ""

                # Year
                year = ""
                pub_date = journal_elem.find("JournalIssue/PubDate") if journal_elem is not None else None
                if pub_date is not None:
                    y = pub_date.find("Year")
                    if y is not None and y.text and y.text.strip().isdigit():
                        year = y.text.strip()
                    else:
                        md = pub_date.find("MedlineDate")
                        if md is not None and md.text:
                            for tok in md.text.split():
                                if tok.isdigit() and len(tok) == 4:
                                    year = tok
                                    break

                # Volume/Issue/Pages
                vol = journal_elem.find("JournalIssue/Volume") if journal_elem is not None else None
                iss = journal_elem.find("JournalIssue/Issue") if journal_elem is not None else None
                volume = vol.text.strip() if vol is not None and vol.text else ""
                issue = iss.text.strip() if iss is not None and iss.text else ""

                medline_pg = art.find("Pagination/MedlinePgn")
                pages = medline_pg.text.strip() if medline_pg is not None and medline_pg.text else ""

                # Authors
                author_list = art.findall("AuthorList/Author")
                authors = []
                first_author_lastname = ""
                for idx, au in enumerate(author_list):
                    last = au.find("LastName")
                    fore = au.find("ForeName")
                    initials = au.find("Initials")

                    if last is None or not last.text:
                        continue
                    last_name = last.text.strip()
                    if idx == 0:
                        first_author_lastname = last_name

                    if fore is not None and fore.text:
                        first_name = fore.text.strip()
                    elif initials is not None and initials.text:
                        first_name = initials.text.strip()
                    else:
                        first_name = ""

                    authors.append(f"{last_name}, {first_name}" if first_name else last_name)

                authors_str = " and ".join(authors)

                # DOI
                doi = ""
                for aid in article.findall(".//ArticleIdList/ArticleId"):
                    if aid.get("IdType") == "doi" and aid.text:
                        doi = aid.text.strip()
                        break

                # BibTeX key
                key_author = first_author_lastname if first_author_lastname else "UnknownAuthor"
                key_year = year if year else "n.d."
                journal_abbrev = journal.replace(" ", "")[:20] if journal else "UnknownJournal"
                bib_key = f"{key_author}{key_year}_{journal_abbrev}"
                bib_key = "".join(ch for ch in bib_key if ch.isalnum() or ch in "_.-")

                def esc(s: str) -> str:
                    return (s or "").replace("{", "").replace("}", "")

                bib_lines = [
                    f"@article{{{bib_key},",
                    f"  author  = {{{esc(authors_str)}}},",
                    f"  title   = {{{esc(title)}}},",
                    f"  journal = {{{esc(journal)}}},",
                    f"  year    = {{{year}}},",
                    f"  volume  = {{{esc(volume)}}},",
                    f"  number  = {{{esc(issue)}}},",
                    f"  pages   = {{{esc(pages)}}},",
                    f"  doi     = {{{esc(doi)}}},",
                    f"  pmid    = {{{pmid}}}",
                    "}"
                ]
                bibtex_str = "\n".join(bib_lines)

                out[pmid] = {
                    "Title": title,
                    "Journal": journal,
                    "Year": year,
                    "FirstAuthor": first_author_lastname,
                    "DOI": doi,
                    "Volume": volume,
                    "Issue": issue,
                    "Pages": pages,
                    "BibTeX_Key": bib_key,
                    "BibTeX": bibtex_str,
                }

            except Exception as e:
                print(f"⚠ Error parsing EFetch record: {e}")
                continue

        time.sleep(SLEEP_SECS)

    return out


def pick_earliest_pmid(pmids, details_dict):
    """Pick PMID with earliest numeric year; missing/invalid years treated as 9999."""
    if not pmids:
        return None

    def year_as_int(pmid):
        y = details_dict.get(pmid, {}).get("Year", "")
        try:
            return int(y)
        except Exception:
            return 9999

    return min(pmids, key=year_as_int)


# -------------------- MAIN --------------------
def main():
    if not NCBI_EMAIL or NCBI_EMAIL == "your_email@domain.com":
        print("❌ Please set your real email address in NCBI_EMAIL.", file=sys.stderr)
        sys.exit(1)

    terms = read_terms(ONTOLOGY_FILE)
    print(f"📄 Loaded {len(terms)} ontology terms from {ONTOLOGY_FILE}\n")

    out_rows = []

    for i, term in enumerate(terms, 1):
        mode = "HIV/SIV + ontology context" if USE_DOMAIN_CONTEXT else "ontology/controlled vocabulary context"
        print(f"[{i}/{len(terms)}] {term} → PubMed search ({mode})")

        query = build_query(term)

        try:
            pmids = esearch_pmids(query, retmax=RETMAX)
            time.sleep(SLEEP_SECS)

            if not pmids:
                print("   ⚠ No PubMed hits.")
                out_rows.append({
                    "Ontology": term,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "No PubMed results for this ontology term under current query constraints",
                })
                continue

            details = efetch_details(pmids)
            earliest_pmid = pick_earliest_pmid(pmids, details)

            if earliest_pmid is None or earliest_pmid not in details:
                print("   ⚠ PMIDs found but metadata missing; writing NA row.")
                out_rows.append({
                    "Ontology": term,
                    "PMID": "NA",
                    "Title": "NA",
                    "Journal": "NA",
                    "Year": "NA",
                    "DOI": "NA",
                    "PubMed_URL": "NA",
                    "BibTeX_Key": "NA",
                    "BibTeX": "NA",
                    "Query": query,
                    "Note": "PMIDs found but metadata missing",
                })
                continue

            meta = details[earliest_pmid]
            print(f"   ✅ Earliest paper: PMID {earliest_pmid}, Year {meta.get('Year','')}")

            out_rows.append({
                "Ontology": term,
                "PMID": earliest_pmid,
                "Title": meta.get("Title", ""),
                "Journal": meta.get("Journal", ""),
                "Year": meta.get("Year", ""),
                "DOI": meta.get("DOI", ""),
                "PubMed_URL": pubmed_url(earliest_pmid),
                "BibTeX_Key": meta.get("BibTeX_Key", ""),
                "BibTeX": meta.get("BibTeX", ""),
                "Query": query,
                "Note": "",
            })

        except Exception as e:
            print(f"   ❌ Error for term '{term}': {e}")
            out_rows.append({
                "Ontology": term,
                "PMID": "NA",
                "Title": "NA",
                "Journal": "NA",
                "Year": "NA",
                "DOI": "NA",
                "PubMed_URL": "NA",
                "BibTeX_Key": "NA",
                "BibTeX": "NA",
                "Query": query,
                "Note": f"Error: {e}",
            })

        time.sleep(SLEEP_SECS)

    cols = [
        "Ontology", "PMID", "Title", "Journal", "Year",
        "DOI", "PubMed_URL", "BibTeX_Key", "BibTeX",
        "Query", "Note"
    ]

    # Write CSV
    with open(OUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=cols, quoting=csv.QUOTE_ALL)
        w.writeheader()
        for row in out_rows:
            safe_row = {k: (str(v).replace('"', "'") if isinstance(v, str) else v) for k, v in row.items()}
            w.writerow(safe_row)

    # Write XLSX
    df = pd.DataFrame(out_rows, columns=cols)
    df.to_excel(OUT_XLSX, index=False)

    print("\n✅ Completed!")
    print(f"Saved CSV : {OUT_CSV}")
    print(f"Saved XLSX: {OUT_XLSX}")
    print(f"Domain context enabled: {USE_DOMAIN_CONTEXT}")


if __name__ == "__main__":
    main()


📄 Loaded 144 ontology terms from /content/start_preart.txt

[1/144] KEGG → PubMed search (ontology/controlled vocabulary context)
   ✅ Earliest paper: PMID 15817693, Year 2005
[2/144] Hepatitis C → PubMed search (ontology/controlled vocabulary context)
   ✅ Earliest paper: PMID 1965321, Year 1990
[3/144] Influenza A → PubMed search (ontology/controlled vocabulary context)
   ✅ Earliest paper: PMID 5316848, Year 1971
[4/144] Measles → PubMed search (ontology/controlled vocabulary context)
   ✅ Earliest paper: PMID 5767931, Year 1969
[5/144] Coronavirus disease → PubMed search (ontology/controlled vocabulary context)
   ✅ Earliest paper: PMID 32123347, Year 2020
[6/144] Transcriptional misregulation in cancer → PubMed search (ontology/controlled vocabulary context)
   ✅ Earliest paper: PMID 25523932, Year 2014
[7/144] RIG-I-like receptor signaling pathway → PubMed search (ontology/controlled vocabulary context)
   ✅ Earliest paper: PMID 26376480, Year 2015
[8/144] Epstein-Barr virus infe